In [1]:
# Install required libraries for loading the EnterpriseRAG-Bench dataset
!pip install -q datasets huggingface_hub

In [2]:
# Select the test split and inspect document structure
from datasets import load_dataset

dataset = load_dataset(
    "ingalepratap/EnterpriseRAG-Bench",
    "documents"
)

print(dataset)


README.md:   0%|          | 0.00/6.93k [00:00<?, ?B/s]

data/documents/test.parquet: reconstructing file:   0%|          |  0.00B / 1.41GB            

data/documents/test.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/511962 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['doc_id', 'source_type', 'title', 'content'],
        num_rows: 511962
    })
})


In [3]:
# Select the test split and inspect document structure
docs = dataset["test"]

print("Number of documents:", len(docs))
print("Columns:", docs.column_names)

print("\nFirst document:")
print(docs[0])

Number of documents: 511962
Columns: ['doc_id', 'source_type', 'title', 'content']

First document:
{'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5', 'source_type': 'confluence', 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)', 'content': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deplo

In [4]:
# Analyze the distribution of documents across enterprise data sources
from collections import Counter

source_counts = Counter(docs["source_type"])

for source, count in source_counts.items():
    print(source, ":", count)

confluence : 5189
fireflies : 10173
github : 8052
gmail : 121390
google_drive : 25108
hubspot : 15017
jira : 6120
linear : 35308
slack : 285605


In [5]:
# Analyze the distribution of documents across enterprise data sources
from datasets import load_dataset

questions_ds = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions"
)

questions = questions_ds["test"]

print("Number of questions:", len(questions))
print("Columns:", questions.column_names)
print("\nFirst question:")
print(questions[0])

README.md:   0%|          | 0.00/6.93k [00:00<?, ?B/s]

data/questions/test.parquet: reconstructing file:   0%|          |  0.00B /  409kB            

data/questions/test.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Number of questions: 500
Columns: ['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

First question:
{'question_id': 'qst_0001', 'question_type': 'basic', 'source_types': ['github'], 'question': 'What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?', 'expected_doc_ids': ['dsid_ae068ee4aa9640159427cd941bef0238'], 'gold_answer': 'The default limits are 10 MiB per file (max_file_size) and 50 MiB total per request (max_total_request_size) for multipart uploads on the OpenAI-compatible endpoints.', 'answer_facts': ['The default per file upload size limit (max_file_size) for multipart uploads on OpenAI-compatible endpoints is 10 MiB.', 'The default total request size limit (max_total_request_size) for multipart uploads on OpenAI-compatible endpoints is 50 MiB.']}


In [6]:
# Collect all document IDs referenced by the 500 questions

all_doc_ids = set()

for q in questions:
    doc_ids = q["expected_doc_ids"]

    if doc_ids:
        all_doc_ids.update(doc_ids)

print("Total unique relevant documents:", len(all_doc_ids))

Total unique relevant documents: 722


In [7]:
# Analyze the different types of questions in the benchmark
from collections import Counter

question_types = Counter(questions["question_type"])

print("Question types:")
for qtype, count in question_types.items():
    print(f"{qtype}: {count}")

Question types:
basic: 175
semantic: 125
intra_document_reasoning: 40
project_related: 40
constrained: 30
conflicting_info: 20
completeness: 20
miscellaneous: 20
high_level: 10
info_not_found: 20


In [8]:
# Analyze the enterprise data sources referenced by the benchmark questions
source_types = Counter()

for q in questions:
    for source in q["source_types"]:
        source_types[source] += 1

print("\nSource types used by questions:")
for source, count in source_types.items():
    print(f"{source}: {count}")


Source types used by questions:
github: 60
linear: 58
fireflies: 25
gmail: 55
google_drive: 60
confluence: 114
jira: 100
hubspot: 34
slack: 79


In [9]:
# Create a set for fast lookup
target_doc_ids = set(all_doc_ids)

# Extract only the 722 relevant documents
memora_docs = docs.filter(
    lambda x: x["doc_id"] in target_doc_ids
)

print("MEMORA documents extracted:", len(memora_docs))
print("Columns:", memora_docs.column_names)

Filter:   0%|          | 0/511962 [00:00<?, ? examples/s]

MEMORA documents extracted: 723
Columns: ['doc_id', 'source_type', 'title', 'content']


In [10]:
# Analyze the source distribution of the selected MEMORA documents
from collections import Counter

memora_source_counts = Counter(memora_docs["source_type"])

print("MEMORA Dataset Source Distribution:\n")

for source, count in memora_source_counts.items():
    print(f"{source}: {count}")

MEMORA Dataset Source Distribution:

confluence: 214
fireflies: 31
github: 65
gmail: 58
google_drive: 63
hubspot: 34
jira: 119
linear: 60
slack: 79


In [11]:
# Inspect sample documents to verify their ID, source, title, and content
for i in range(3):
    print("=" * 80)
    print("DOCUMENT", i + 1)
    print("=" * 80)

    print("Document ID:", memora_docs[i]["doc_id"])
    print("Source:", memora_docs[i]["source_type"])
    print("Title:", memora_docs[i]["title"])
    print("\nContent:")
    print(memora_docs[i]["content"][:2000])
    print()

DOCUMENT 1
Document ID: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c
Source: confluence
Title: Support Process: Customer Communications Requirements for Model Rollouts

Content:
h1. Purpose
This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules.

h1. Who must be notified

h2. Always notify
* Any Dedicated customer receiving the new version as default
* Any Private customer (VPC/on-prem) impacted by the change

h2. Notify when material behavior changes
For Hosted API customers, notify when any of the following change materially for a commonly used alias:
* output format or tool-calling behavior
* maximum context/output token limits
* latency or reliability expectations
* safety behaviors (refusals, content filtering)

h1. Timing
* Dedicated: at least 5 business days before the change window, unless the customer agrees otherwise.
* Private: per customer 

In [12]:
# Save the selected dataset in Parquet format
memora_docs.to_parquet("MEMORA_raw_documents.parquet")

print("Saved successfully!")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved successfully!


In [13]:
memora_docs.to_json(
    "MEMORA_raw_documents.jsonl"
)

print("JSONL saved successfully!")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

JSONL saved successfully!


In [14]:
# Check basic information about our MEMORA dataset

print("Total documents:", len(memora_docs))
print("Columns:", memora_docs.column_names)

print("\nMissing values:")

for column in memora_docs.column_names:
    missing = sum(
        1 for value in memora_docs[column]
        if value is None or str(value).strip() == ""
    )
    print(f"{column}: {missing}")

Total documents: 723
Columns: ['doc_id', 'source_type', 'title', 'content']

Missing values:
doc_id: 0
source_type: 0
title: 0
content: 0


In [15]:
# Calculate document length

doc_lengths = [
    len(str(text))
    for text in memora_docs["content"]
]

print("Shortest document:", min(doc_lengths), "characters")
print("Longest document:", max(doc_lengths), "characters")
print("Average document:", sum(doc_lengths) // len(doc_lengths), "characters")

Shortest document: 531 characters
Longest document: 32936 characters
Average document: 6546 characters


In [16]:
# Inspect sample documents before applying text preprocessing
for i in range(5):

    print("\n" + "=" * 80)
    print(f"DOCUMENT {i + 1}")
    print("=" * 80)

    print("ID:", memora_docs[i]["doc_id"])
    print("SOURCE:", memora_docs[i]["source_type"])
    print("TITLE:", memora_docs[i]["title"])

    print("\nCONTENT:")
    print(memora_docs[i]["content"][:1500])


DOCUMENT 1
ID: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c
SOURCE: confluence
TITLE: Support Process: Customer Communications Requirements for Model Rollouts

CONTENT:
h1. Purpose
This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules.

h1. Who must be notified

h2. Always notify
* Any Dedicated customer receiving the new version as default
* Any Private customer (VPC/on-prem) impacted by the change

h2. Notify when material behavior changes
For Hosted API customers, notify when any of the following change materially for a commonly used alias:
* output format or tool-calling behavior
* maximum context/output token limits
* latency or reliability expectations
* safety behaviors (refusals, content filtering)

h1. Timing
* Dedicated: at least 5 business days before the change window, unless the customer agrees otherwise.
* Private: per customer contract

In [17]:
# Import libraries required for text cleaning and data processing
# Define a text cleaning function to normalize enterprise document content
import re
import pandas as pd

# Apply text cleaning to document titles and content
def clean_text(text):
    if text is None:
        return ""

    text = str(text)

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove excessive spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Clean Markdown/Confluence heading markers
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^h[1-6]\.\s*", "", text, flags=re.MULTILINE)

    # Normalize bullet formatting
    text = re.sub(r"^[*+-]\s*", "- ", text, flags=re.MULTILINE)

    # Remove unnecessary trailing spaces
    text = re.sub(r"[ \t]+\n", "\n", text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text


# Create cleaned dataset
cleaned_docs = []

for doc in memora_docs:

    cleaned_content = clean_text(doc["content"])
    cleaned_title = clean_text(doc["title"])

    cleaned_docs.append({
        "doc_id": doc["doc_id"],
        "source_type": doc["source_type"],
        "title": cleaned_title,
        "content": cleaned_content
    })

print("Documents processed:", len(cleaned_docs))

Documents processed: 723


In [18]:
# Convert cleaned documents into a Pandas DataFrame for analysis
df = pd.DataFrame(cleaned_docs)

print(df.shape)
print(df.head())

(723, 4)
                                  doc_id source_type  \
0  dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c  confluence   
1  dsid_9c8cb10d65324c03a8ededbd98c1b058  confluence   
2  dsid_91abfa2001d747b3b570e8abce00ef4d  confluence   
3  dsid_dd0fe40a2c71489793d566e46e95bb93  confluence   
4  dsid_0b129c86cc6240b088a137d6634eef25  confluence   

                                               title  \
0  Support Process: Customer Communications Requi...   
1  Go-Live Protocols: Cross-Org Communications & ...   
2  Support Ops: Reconciling token accounting disc...   
3  SOP: intake and coordination for customer-mana...   
4  Slack Bridge and Incident Channel Conventions ...   

                                             content  
0  Purpose\nThis page defines the minimum custome...  
1  Summary\n- ------\nThis document defines the c...  
2  Purpose\nThis page documents how Support shoul...  
3  Purpose\nThis SOP describes how Customer Succe...  
4  Purpose\nThis page standardizes **how S

In [19]:
# Convert cleaned documents into a Pandas DataFrame for analysis
print("BEFORE:")
print(memora_docs[0]["content"][:1000])

print("\n" + "="*80)

print("AFTER:")
print(df.iloc[0]["content"][:1000])

BEFORE:
h1. Purpose
This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules.

h1. Who must be notified

h2. Always notify
* Any Dedicated customer receiving the new version as default
* Any Private customer (VPC/on-prem) impacted by the change

h2. Notify when material behavior changes
For Hosted API customers, notify when any of the following change materially for a commonly used alias:
* output format or tool-calling behavior
* maximum context/output token limits
* latency or reliability expectations
* safety behaviors (refusals, content filtering)

h1. Timing
* Dedicated: at least 5 business days before the change window, unless the customer agrees otherwise.
* Private: per customer contract; default to 10 business days prior to the maintenance window.
* Hosted: at least 72 hours prior for material behavior changes; otherwise release notes are

In [20]:
# Add character count, word count, and source metadata for analysis
df["char_count"] = df["content"].str.len()

df["word_count"] = df["content"].str.split().str.len()

df["source"] = df["source_type"]

print(df[[
    "doc_id",
    "source",
    "title",
    "char_count",
    "word_count"
]].head())

                                  doc_id      source  \
0  dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c  confluence   
1  dsid_9c8cb10d65324c03a8ededbd98c1b058  confluence   
2  dsid_91abfa2001d747b3b570e8abce00ef4d  confluence   
3  dsid_dd0fe40a2c71489793d566e46e95bb93  confluence   
4  dsid_0b129c86cc6240b088a137d6634eef25  confluence   

                                               title  char_count  word_count  
0  Support Process: Customer Communications Requi...        2380         365  
1  Go-Live Protocols: Cross-Org Communications & ...        9074        1204  
2  Support Ops: Reconciling token accounting disc...        1975         291  
3  SOP: intake and coordination for customer-mana...        2984         439  
4  Slack Bridge and Incident Channel Conventions ...       10303        1502  


In [21]:
# Save the cleaned MEMORA dataset in reusable formats
df.to_parquet(
    "MEMORA_cleaned_documents.parquet",
    index=False
)

df.to_json(
    "MEMORA_cleaned_documents.jsonl",
    orient="records",
    lines=True
)

print("MEMORA cleaned dataset saved successfully.")

MEMORA cleaned dataset saved successfully.


In [22]:
# Define chunk size and overlap for semantic retrieval
import re
import pandas as pd

CHUNK_SIZE = 500
CHUNK_OVERLAP = 75


def split_into_chunks(text, chunk_size=500, overlap=75):
    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunk_records = []
# Split each document into overlapping word-based chunks
for _, row in df.iterrows():

    chunks = split_into_chunks(
        row["content"],
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    for i, chunk in enumerate(chunks):
    # Create metadata for every generated chunk

        chunk_records.append({
            "chunk_id": f'{row["doc_id"]}_chunk_{i+1}',
            "doc_id": row["doc_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "chunk_index": i + 1,
            "content": chunk,
            "word_count": len(chunk.split())
        })


chunks_df = pd.DataFrame(chunk_records)

print("Documents:", len(df))
print("Total chunks:", len(chunks_df))

print("\nColumns:")
print(chunks_df.columns.tolist())

print("\nFirst chunk:")
print(chunks_df.iloc[0])

Documents: 723
Total chunks: 1785

Columns:
['chunk_id', 'doc_id', 'source_type', 'title', 'chunk_index', 'content', 'word_count']

First chunk:
chunk_id           dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1
doc_id                     dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c
source_type                                           confluence
title          Support Process: Customer Communications Requi...
chunk_index                                                    1
content        Purpose This page defines the minimum customer...
word_count                                                   365
Name: 0, dtype: object


In [23]:
print("Average chunks per document:",
      round(len(chunks_df) / len(df), 2))

print("\nChunk size statistics:")

print(chunks_df["word_count"].describe())

Average chunks per document: 2.47

Chunk size statistics:
count    1785.000000
mean      410.229132
std       133.251417
min        76.000000
25%       328.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: word_count, dtype: float64


In [24]:
chunks_df.to_parquet(
    "MEMORA_chunks.parquet",
    index=False
)

chunks_df.to_json(
    "MEMORA_chunks.jsonl",
    orient="records",
    lines=True
)

print("MEMORA chunked dataset saved successfully.")

MEMORA chunked dataset saved successfully.


In [25]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75

def split_into_chunks(text, chunk_size=500, overlap=75):
    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunk_records = []

for _, row in df.iterrows():

    chunks = split_into_chunks(
        row["content"],
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    for i, chunk in enumerate(chunks):

        chunk_records.append({
            "chunk_id": f"{row['doc_id']}_chunk_{i+1}",
            "doc_id": row["doc_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "chunk_index": i + 1,
            "content": chunk,
            "word_count": len(chunk.split())
        })


chunks_df = pd.DataFrame(chunk_records)

print("Original documents:", len(df))
print("Total chunks:", len(chunks_df))
print("Average chunks/document:",
      round(len(chunks_df) / len(df), 2))

Original documents: 723
Total chunks: 1785
Average chunks/document: 2.47


In [26]:
print(chunks_df.head(3).to_string())

                                        chunk_id                                 doc_id source_type                                                                     title  chunk_index                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [27]:
print(chunks_df["word_count"].describe())

count    1785.000000
mean      410.229132
std       133.251417
min        76.000000
25%       328.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: word_count, dtype: float64


In [28]:
chunks_df.to_parquet(
    "MEMORA_chunks.parquet",
    index=False
)

chunks_df.to_json(
    "MEMORA_chunks.jsonl",
    orient="records",
    lines=True
)

print("Chunked dataset saved successfully.")

Chunked dataset saved successfully.


In [29]:
!pip install -q sentence-transformers

In [30]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [31]:
# Convert all document chunks into normalized semantic embeddings
import numpy as np

texts = chunks_df["content"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Embedding shape: (1785, 384)


In [32]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.7 MB/s eta 0:00:00


In [33]:
# Convert embeddings to float32 format required by FAISS
import faiss
import numpy as np

# Convert embeddings to float32
embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

# Number of dimensions
dimension = embedding_matrix.shape[1]

# Create cosine-similarity index
index = faiss.IndexFlatIP(dimension)

# Add embeddings
index.add(embedding_matrix)

print("FAISS index created successfully.")
print("Number of vectors:", index.ntotal)
print("Vector dimensions:", dimension)

FAISS index created successfully.
Number of vectors: 1785
Vector dimensions: 384


In [34]:
faiss.write_index(
    index,
    "MEMORA_faiss.index"
)

np.save(
    "MEMORA_embeddings.npy",
    embedding_matrix
)

print("FAISS index and embeddings saved successfully.")

FAISS index and embeddings saved successfully.


In [35]:
# Define a natural-language enterprise knowledge query
query = "What are the requirements for customer communication during a model rollout?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(
    query_embedding,
    5
)

print("Query:", query)
print("\nTop 5 Results:\n")

for rank, idx in enumerate(indices[0], start=1):

    print("=" * 80)
    print("Rank:", rank)
    print("Similarity:", round(float(scores[0][rank-1]), 4))
    print("Source:", chunks_df.iloc[idx]["source_type"])
    print("Title:", chunks_df.iloc[idx]["title"])
    print("Document ID:", chunks_df.iloc[idx]["doc_id"])
    print("\nContent:")
    print(chunks_df.iloc[idx]["content"][:1000])

Query: What are the requirements for customer communication during a model rollout?

Top 5 Results:

Rank: 1
Similarity: 0.5839
Source: confluence
Title: Support Process: Customer Communications Requirements for Model Rollouts
Document ID: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c

Content:
Purpose This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules. Who must be notified Always notify - Any Dedicated customer receiving the new version as default - Any Private customer (VPC/on-prem) impacted by the change Notify when material behavior changes For Hosted API customers, notify when any of the following change materially for a commonly used alias: - output format or tool-calling behavior - maximum context/output token limits - latency or reliability expectations - safety behaviors (refusals, content filtering) Timing - Dedicated: at least 5 business 

In [36]:
# Retrieve the top 5 most semantically similar document chunks
def retrieve_documents(query, top_k=5):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []
# Display retrieved documents with similarity scores and metadata
    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "doc_id": chunks_df.iloc[idx]["doc_id"],
            "source_type": chunks_df.iloc[idx]["source_type"],
            "title": chunks_df.iloc[idx]["title"],
            "content": chunks_df.iloc[idx]["content"]
        })

    return results

In [37]:
# Define a reusable function for semantic document retrieval
query = "Who is responsible for handling customer escalations?"

results = retrieve_documents(query, top_k=5)

for result in results:

    print("=" * 80)
    print("Rank:", result["rank"])
    print("Score:", round(result["score"], 4))
    print("Source:", result["source_type"])
    print("Title:", result["title"])
    print("Document:", result["doc_id"])
    print("\n", result["content"][:800])

Rank: 1
Score: 0.5844
Source: confluence
Title: Slack Bridge and Incident Channel Conventions (Support Escalations)
Document: dsid_0b129c86cc6240b088a137d6634eef25

 Purpose This page standardizes **how Support starts and runs a Slack bridge** for customer-impacting escalations, and how/when we transition the conversation into **#incidents**. Goals: - Reduce time-to-triage by ensuring every bridge starts with complete context. - Ensure Engineering/SRE can take action quickly without back-and-forth. - Keep Jira (SUP tickets) and Slack synchronized so escalations are discoverable after the bridge ends. - Ensure **one owner** is accountable for customer-facing updates (email / ticket / status page). Non-goals: - Replace the company incident process. If an issue meets incident criteria, we transition to #incidents and follow the incident playbook. - -- Definitions - **Support bridge:** A short-lived Slack channel created by Support to coordinate triage/mit
Rank: 2
Score: 0.5746
Source: con

In [38]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [39]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy loaded successfully!")

spaCy loaded successfully!


In [40]:
# Test entity extraction on one existing MEMORA chunk

text = chunks_df.iloc[0]["content"]

doc = nlp(text)

print("Sample text:")
print(text[:1000])

print("\nEntities found:")
for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Sample text:
Purpose This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules. Who must be notified Always notify - Any Dedicated customer receiving the new version as default - Any Private customer (VPC/on-prem) impacted by the change Notify when material behavior changes For Hosted API customers, notify when any of the following change materially for a commonly used alias: - output format or tool-calling behavior - maximum context/output token limits - latency or reliability expectations - safety behaviors (refusals, content filtering) Timing - Dedicated: at least 5 business days before the change window, unless the customer agrees otherwise. - Private: per customer contract; default to 10 business days prior to the maintenance window. - Hosted: at least 72 hours prior for material behavior changes; otherwise release notes are sufficient. Requir

In [41]:
# Inspect spaCy entities with surrounding context

for ent in doc.ents:
    start = max(0, ent.start_char - 50)
    end = min(len(text), ent.end_char + 50)

    print("-" * 80)
    print("Entity :", ent.text)
    print("Label  :", ent.label_)
    print("Context:", text[start:end])

--------------------------------------------------------------------------------
Entity : Customer
Label  : ORG
Context: el version rollouts. Rollout plans MUST include a Customer communications section that follows these rules. 
--------------------------------------------------------------------------------
Entity : Notify
Label  : ORG
Context: ate customer (VPC/on-prem) impacted by the change Notify when material behavior changes For Hosted API cus
--------------------------------------------------------------------------------
Entity : at least 5
Label  : CARDINAL
Context: (refusals, content filtering) Timing - Dedicated: at least 5 business days before the change window, unless th
--------------------------------------------------------------------------------
Entity : 10 business days
Label  : DATE
Context: ise. - Private: per customer contract; default to 10 business days prior to the maintenance window. - Hosted: at lea
----------------------------------------------------------

In [42]:
# MEMORA-specific entity types

MEMORA_ENTITY_TYPES = [
    "PERSON",
    "ORGANIZATION",
    "PROJECT",
    "TEAM",
    "ROLE",
    "TECHNOLOGY",
    "PRODUCT",
    "DOCUMENT",
    "CUSTOMER",
    "DECISION",
    "POLICY",
    "TIME",
    "LOCATION"
]

print("MEMORA Entity Types:")
for entity_type in MEMORA_ENTITY_TYPES:
    print("-", entity_type)

MEMORA Entity Types:
- PERSON
- ORGANIZATION
- PROJECT
- TEAM
- ROLE
- TECHNOLOGY
- PRODUCT
- DOCUMENT
- CUSTOMER
- DECISION
- POLICY
- TIME
- LOCATION


In [43]:
# MEMORA Entity Extraction

def extract_entities(text):
    """
    Extract entities from a MEMORA text chunk using spaCy.
    """

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "spacy_label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

    return entities


# Test on the first MEMORA chunk
sample_text = chunks_df.iloc[0]["content"]

entities = extract_entities(sample_text)

print("Entities extracted:", len(entities))

for entity in entities:
    print(entity)

Entities extracted: 11
{'text': 'Customer', 'spacy_label': 'ORG', 'start': 149, 'end': 157}
{'text': 'Notify', 'spacy_label': 'ORG', 'start': 364, 'end': 370}
{'text': 'at least 5', 'spacy_label': 'CARDINAL', 'start': 692, 'end': 702}
{'text': '10 business days', 'spacy_label': 'DATE', 'start': 826, 'end': 842}
{'text': 'at least 72 hours', 'spacy_label': 'TIME', 'start': 886, 'end': 903}
{'text': 'Channels - Dedicated/Private', 'spacy_label': 'ORG', 'start': 1421, 'end': 1449}
{'text': 'Approvals Customer', 'spacy_label': 'ORG', 'start': 1662, 'end': 1680}
{'text': 'Customer Success', 'spacy_label': 'ORG', 'start': 1719, 'end': 1735}
{'text': 'Templates', 'spacy_label': 'PERSON', 'start': 1869, 'end': 1878}
{'text': 'Redwood Inference Opt', 'spacy_label': 'ORG', 'start': 2016, 'end': 2037}
{'text': 'at least 30 days', 'spacy_label': 'DATE', 'start': 2163, 'end': 2179}


In [44]:
# MEMORA Entity Classification

def classify_entity(entity, text):
    """
    Convert spaCy entity labels into MEMORA-specific entity types.
    """

    entity_text = entity["text"]
    spacy_label = entity["spacy_label"]

    text_lower = entity_text.lower()

    # -------------------------
    # TIME
    # -------------------------
    if spacy_label in ["DATE", "TIME", "CARDINAL"]:
        if any(word in text_lower for word in [
            "day", "days", "hour", "hours", "week", "weeks",
            "month", "months", "year", "years", "minute", "minutes"
        ]):
            return "TIME"

    # -------------------------
    # TEAM
    # -------------------------
    if entity_text.lower() in [
        "customer success"
    ]:
        return "TEAM"

    # -------------------------
    # CUSTOMER
    # -------------------------
    if entity_text.lower() == "customer":
        return "CUSTOMER"

    # -------------------------
    # TECHNOLOGY / PRODUCT
    # -------------------------
    if entity_text.lower() in [
        "redwood inference opt"
    ]:
        return "PRODUCT"

    # -------------------------
    # ORGANIZATION
    # -------------------------
    if spacy_label == "ORG":
        return "ORGANIZATION"

    # -------------------------
    # PERSON
    # -------------------------
    if spacy_label == "PERSON":
        return "PERSON"

    # -------------------------
    # Default
    # -------------------------
    return None


# Test classification on extracted entities

classified_entities = []

for entity in entities:

    memora_type = classify_entity(entity, sample_text)

    classified_entities.append({
        "text": entity["text"],
        "spacy_label": entity["spacy_label"],
        "memora_type": memora_type,
        "start": entity["start"],
        "end": entity["end"]
    })


print("MEMORA Classified Entities:\n")

for entity in classified_entities:
    print(entity)

MEMORA Classified Entities:

{'text': 'Customer', 'spacy_label': 'ORG', 'memora_type': 'CUSTOMER', 'start': 149, 'end': 157}
{'text': 'Notify', 'spacy_label': 'ORG', 'memora_type': 'ORGANIZATION', 'start': 364, 'end': 370}
{'text': 'at least 5', 'spacy_label': 'CARDINAL', 'memora_type': None, 'start': 692, 'end': 702}
{'text': '10 business days', 'spacy_label': 'DATE', 'memora_type': 'TIME', 'start': 826, 'end': 842}
{'text': 'at least 72 hours', 'spacy_label': 'TIME', 'memora_type': 'TIME', 'start': 886, 'end': 903}
{'text': 'Channels - Dedicated/Private', 'spacy_label': 'ORG', 'memora_type': 'ORGANIZATION', 'start': 1421, 'end': 1449}
{'text': 'Approvals Customer', 'spacy_label': 'ORG', 'memora_type': 'ORGANIZATION', 'start': 1662, 'end': 1680}
{'text': 'Customer Success', 'spacy_label': 'ORG', 'memora_type': 'TEAM', 'start': 1719, 'end': 1735}
{'text': 'Templates', 'spacy_label': 'PERSON', 'memora_type': 'PERSON', 'start': 1869, 'end': 1878}
{'text': 'Redwood Inference Opt', 'spacy_

In [45]:
# Improved MEMORA Entity Classification + False Positive Filtering

def classify_entity(entity, text):
    entity_text = entity["text"]
    spacy_label = entity["spacy_label"]

    text_lower = entity_text.lower()

    # Get surrounding context
    start = max(0, entity["start"] - 50)
    end = min(len(text), entity["end"] + 50)
    context = text[start:end].lower()

    # -------------------------
    # FALSE POSITIVE FILTERS
    # -------------------------
    false_positives = [
        "notify",
        "templates",
        "channels - dedicated/private",
        "approvals customer"
    ]

    if text_lower in false_positives:
        return None

    # -------------------------
    # TIME
    # -------------------------
    if spacy_label in ["DATE", "TIME"]:
        return "TIME"

    # CARDINAL numbers followed by time units
    if spacy_label == "CARDINAL":
        time_words = [
            "day", "days",
            "hour", "hours",
            "week", "weeks",
            "month", "months",
            "year", "years",
            "minute", "minutes"
        ]

        if any(word in context for word in time_words):
            return "TIME"

    # -------------------------
    # CUSTOMER
    # -------------------------
    if text_lower == "customer":
        return "CUSTOMER"

    # -------------------------
    # TEAM
    # -------------------------
    if text_lower == "customer success":
        return "TEAM"

    # -------------------------
    # PRODUCT
    # -------------------------
    if text_lower == "redwood inference opt":
        return "PRODUCT"

    # -------------------------
    # ORGANIZATION
    # -------------------------
    if spacy_label == "ORG":
        return "ORGANIZATION"

    # -------------------------
    # PERSON
    # -------------------------
    if spacy_label == "PERSON":
        return "PERSON"

    return None


# Apply improved classification

classified_entities = []

for entity in entities:

    memora_type = classify_entity(
        entity,
        sample_text
    )

    if memora_type is not None:
        classified_entities.append({
            "text": entity["text"],
            "spacy_label": entity["spacy_label"],
            "memora_type": memora_type,
            "start": entity["start"],
            "end": entity["end"]
        })


print("Filtered MEMORA Entities:\n")

for entity in classified_entities:
    print(
        entity["text"],
        "->",
        entity["memora_type"]
    )

print("\nTotal valid entities:", len(classified_entities))

Filtered MEMORA Entities:

Customer -> CUSTOMER
at least 5 -> TIME
10 business days -> TIME
at least 72 hours -> TIME
Customer Success -> TEAM
Redwood Inference Opt -> PRODUCT
at least 30 days -> TIME

Total valid entities: 7


In [46]:
# MEMORA Entity Normalization

def normalize_entity(entity_text):
    """
    Normalize an entity mention for consistent storage and comparison.
    """

    normalized = entity_text.strip().lower()

    # Replace multiple spaces with a single space
    normalized = " ".join(normalized.split())

    return normalized


# Test normalization

normalized_entities = []

for entity in classified_entities:

    normalized = normalize_entity(entity["text"])

    normalized_entities.append({
        "original": entity["text"],
        "normalized": normalized,
        "memora_type": entity["memora_type"]
    })


print("Normalized MEMORA Entities:\n")

for entity in normalized_entities:
    print(
        entity["original"],
        "->",
        entity["normalized"],
        "->",
        entity["memora_type"]
    )

Normalized MEMORA Entities:

Customer -> customer -> CUSTOMER
at least 5 -> at least 5 -> TIME
10 business days -> 10 business days -> TIME
at least 72 hours -> at least 72 hours -> TIME
Customer Success -> customer success -> TEAM
Redwood Inference Opt -> redwood inference opt -> PRODUCT
at least 30 days -> at least 30 days -> TIME


In [47]:
# MEMORA Entity Deduplication

def deduplicate_entities(entities):
    """
    Remove duplicate entities based on normalized text + MEMORA type.
    """

    unique_entities = []
    seen = set()

    for entity in entities:

        key = (
            entity["normalized"],
            entity["memora_type"]
        )

        if key not in seen:
            seen.add(key)
            unique_entities.append(entity)

    return unique_entities


# Deduplicate the normalized entities

deduplicated_entities = deduplicate_entities(
    normalized_entities
)

print("Deduplicated MEMORA Entities:\n")

for entity in deduplicated_entities:
    print(
        entity["normalized"],
        "->",
        entity["memora_type"]
    )

print("\nBefore deduplication:", len(normalized_entities))
print("After deduplication:", len(deduplicated_entities))

Deduplicated MEMORA Entities:

customer -> CUSTOMER
at least 5 -> TIME
10 business days -> TIME
at least 72 hours -> TIME
customer success -> TEAM
redwood inference opt -> PRODUCT
at least 30 days -> TIME

Before deduplication: 7
After deduplication: 7


In [48]:
# ==========================================
# MEMORA COMPLETE ENTITY EXTRACTION PIPELINE
# ==========================================

def extract_and_process_entities(text):
    """
    Complete MEMORA entity pipeline:
    spaCy extraction
    -> MEMORA classification
    -> filtering
    -> normalization
    -> deduplication
    """

    # 1. Raw spaCy extraction
    doc = nlp(text)

    raw_entities = []

    for ent in doc.ents:
        raw_entities.append({
            "text": ent.text,
            "spacy_label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

    # 2. Classification + filtering
    classified = []

    for entity in raw_entities:

        memora_type = classify_entity(
            entity,
            text
        )

        if memora_type is not None:
            classified.append({
                "text": entity["text"],
                "spacy_label": entity["spacy_label"],
                "memora_type": memora_type,
                "start": entity["start"],
                "end": entity["end"]
            })

    # 3. Normalization
    normalized = []

    for entity in classified:

        normalized.append({
            "original": entity["text"],
            "normalized": normalize_entity(entity["text"]),
            "memora_type": entity["memora_type"],
            "start": entity["start"],
            "end": entity["end"]
        })

    # 4. Deduplication
    final_entities = deduplicate_entities(normalized)

    return final_entities


# ------------------------------------------
# Test on the first chunk
# ------------------------------------------

test_entities = extract_and_process_entities(
    chunks_df.iloc[0]["content"]
)

print("Final entities from first chunk:\n")

for entity in test_entities:
    print(
        entity["original"],
        "->",
        entity["normalized"],
        "->",
        entity["memora_type"]
    )

print("\nTotal final entities:", len(test_entities))

Final entities from first chunk:

Customer -> customer -> CUSTOMER
at least 5 -> at least 5 -> TIME
10 business days -> 10 business days -> TIME
at least 72 hours -> at least 72 hours -> TIME
Customer Success -> customer success -> TEAM
Redwood Inference Opt -> redwood inference opt -> PRODUCT
at least 30 days -> at least 30 days -> TIME

Total final entities: 7


In [49]:
# ==========================================
# RUN ENTITY PIPELINE ON ALL MEMORA CHUNKS
# ==========================================

all_entities = []

for idx, row in chunks_df.iterrows():

    chunk_id = row["chunk_id"]
    text = row["content"]

    entities = extract_and_process_entities(text)

    for entity in entities:

        all_entities.append({
            "chunk_id": chunk_id,
            "entity": entity["original"],
            "normalized_entity": entity["normalized"],
            "entity_type": entity["memora_type"],
            "start": entity["start"],
            "end": entity["end"]
        })


# Convert to DataFrame
entities_df = pd.DataFrame(all_entities)


print("Entity extraction completed!")
print("Total entities:", len(entities_df))
print("Total chunks processed:", chunks_df["chunk_id"].nunique())

print("\nEntity type distribution:")
print(entities_df["entity_type"].value_counts())

print("\nSample entities:")
display(entities_df.head(20))

Entity extraction completed!
Total entities: 28394
Total chunks processed: 1784

Entity type distribution:
entity_type
ORGANIZATION    11522
TIME             8693
PERSON           8023
CUSTOMER          111
TEAM               44
PRODUCT             1
Name: count, dtype: int64

Sample entities:


,chunk_id,entity,normalized_entity,entity_type,start,end
0,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,Customer,customer,CUSTOMER,149,157
1,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,at least 5,at least 5,TIME,692,702
2,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,10 business days,10 business days,TIME,826,842
3,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,at least 72 hours,at least 72 hours,TIME,886,903
4,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,Customer Success,customer success,TEAM,1719,1735
5,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,Redwood Inference Opt,redwood inference opt,PRODUCT,2016,2037
6,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,at least 30 days,at least 30 days,TIME,2163,2179
7,dsid_9c8cb10d65324c03a8ededbd98c1b058_chunk_1,Redwood Inference,redwood inference,ORGANIZATION,89,106
8,dsid_9c8cb10d65324c03a8ededbd98c1b058_chunk_1,the 72 hours,the 72 hours,TIME,321,333
9,dsid_9c8cb10d65324c03a8ededbd98c1b058_chunk_1,72 hours,72 hours,TIME,354,362


In [50]:
# ==========================================
# MEMORA ENTITY QUALITY CHECK
# ==========================================

print("Unique normalized entities by type:\n")

for entity_type in entities_df["entity_type"].unique():

    print(f"\n===== {entity_type} =====")

    subset = (
        entities_df[
            entities_df["entity_type"] == entity_type
        ]["normalized_entity"]
        .value_counts()
        .head(30)
    )

    print(subset)

Unique normalized entities by type:


===== CUSTOMER =====
normalized_entity
customer    111
Name: count, dtype: int64

===== TIME =====
normalized_entity
monthly       126
2             113
today         108
1             102
2026-03-12     89
2026-03        87
3              79
2026-03-10     77
2026           76
30 days        75
10 minutes     72
2026-03-11     66
5              66
4              64
30 minutes     64
48 hours       63
15 minutes     57
weekly         55
7 days         54
quarterly      50
next week      50
90 days        48
daily          48
24 hours       47
2026-03-13     44
one            43
5 minutes      42
docs           39
two            35
30-day         34
Name: count, dtype: int64

===== TEAM =====
normalized_entity
customer success    44
Name: count, dtype: int64

===== PRODUCT =====
normalized_entity
redwood inference opt    1
Name: count, dtype: int64

===== ORGANIZATION =====
normalized_entity
sre         432
api         410
slo         268
kms       

In [51]:
# ==========================================
# MEMORA DOMAIN ENTITY DICTIONARY
# ==========================================

MEMORA_TECHNOLOGY_TERMS = {
    "api",
    "sdk",
    "gpu",
    "kms",
    "s3",
    "cpu",
    "cli",
    "hsm",
    "iam",
    "rbac",
    "sre",
    "slo",
    "ci",
    "vpc",
    "qps",
    "ttl",
    "pii",
    "sse",
    "json",
    "jira",
    "grafana",
    "slack"
}

MEMORA_POLICY_TERMS = {
    "sla"
}

MEMORA_NON_ENTITY_TERMS = {
    "p95",
    "poc",
    "eta",
    "validate",
    "monthly",
    "weekly",
    "daily",
    "quarterly",
    "today",
    "docs",
    "doc",
    "ui",
    "cs",
    "ae"
}


def classify_entity(entity, text):

    entity_text = entity["text"]
    spacy_label = entity["spacy_label"]

    text_lower = entity_text.strip().lower()

    # ======================================
    # NON-ENTITY FILTER
    # ======================================

    if text_lower in MEMORA_NON_ENTITY_TERMS:
        return None

    # ======================================
    # TIME
    # ======================================

    if spacy_label in ["DATE", "TIME"]:
        return "TIME"

    if spacy_label == "CARDINAL":

        time_words = [
            "day", "days",
            "hour", "hours",
            "week", "weeks",
            "month", "months",
            "year", "years",
            "minute", "minutes"
        ]

        start = max(0, entity["start"] - 50)
        end = min(len(text), entity["end"] + 50)

        context = text[start:end].lower()

        if any(word in context for word in time_words):
            return "TIME"

    # ======================================
    # CUSTOMER
    # ======================================

    if text_lower == "customer":
        return "CUSTOMER"

    # ======================================
    # TEAM
    # ======================================

    if text_lower in {
        "customer success",
        "solutions engineering"
    }:
        return "TEAM"

    # ======================================
    # TECHNOLOGY
    # ======================================

    if text_lower in MEMORA_TECHNOLOGY_TERMS:
        return "TECHNOLOGY"

    # ======================================
    # POLICY
    # ======================================

    if text_lower in MEMORA_POLICY_TERMS:
        return "POLICY"

    # ======================================
    # PRODUCT
    # ======================================

    if text_lower == "redwood inference opt":
        return "PRODUCT"

    # ======================================
    # ORGANIZATION
    # ======================================

    if spacy_label == "ORG":
        return "ORGANIZATION"

    # ======================================
    # PERSON
    # ======================================

    if spacy_label == "PERSON":

        # Technical terms incorrectly detected as people
        if text_lower in MEMORA_TECHNOLOGY_TERMS:
            return "TECHNOLOGY"

        return "PERSON"

    return None

In [52]:
# Test improved classifier on first chunk

test_entities = extract_and_process_entities(
    chunks_df.iloc[0]["content"]
)

print("Improved entities from first chunk:\n")

for entity in test_entities:
    print(
        entity["original"],
        "->",
        entity["normalized"],
        "->",
        entity["memora_type"]
    )

print("\nTotal entities:", len(test_entities))

Improved entities from first chunk:

Customer -> customer -> CUSTOMER
Notify -> notify -> ORGANIZATION
at least 5 -> at least 5 -> TIME
10 business days -> 10 business days -> TIME
at least 72 hours -> at least 72 hours -> TIME
Channels - Dedicated/Private -> channels - dedicated/private -> ORGANIZATION
Approvals Customer -> approvals customer -> ORGANIZATION
Customer Success -> customer success -> TEAM
Templates -> templates -> PERSON
Redwood Inference Opt -> redwood inference opt -> PRODUCT
at least 30 days -> at least 30 days -> TIME

Total entities: 11


In [53]:
# ==========================================
# MEMORA FINAL ENTITY PIPELINE - TEST VERSION
# ==========================================

def extract_and_process_entities(text):

    doc = nlp(text)

    processed = []

    # --------------------------------------
    # Raw spaCy extraction
    # --------------------------------------
    for ent in doc.ents:

        entity_text = ent.text.strip()
        text_lower = entity_text.lower()

        # ----------------------------------
        # Explicit false-positive filtering
        # ----------------------------------
        false_positives = {
            "notify",
            "templates",
            "channels - dedicated/private",
            "approvals customer"
        }

        if text_lower in false_positives:
            continue

        # ----------------------------------
        # MEMORA classification
        # ----------------------------------

        memora_type = None

        # TIME
        if ent.label_ in ["DATE", "TIME"]:
            memora_type = "TIME"

        elif ent.label_ == "CARDINAL":

            start = max(0, ent.start_char - 50)
            end = min(len(text), ent.end_char + 50)

            context = text[start:end].lower()

            time_words = [
                "day", "days",
                "hour", "hours",
                "week", "weeks",
                "month", "months",
                "year", "years",
                "minute", "minutes"
            ]

            if any(word in context for word in time_words):
                memora_type = "TIME"

        # CUSTOMER
        if text_lower == "customer":
            memora_type = "CUSTOMER"

        # TEAM
        elif text_lower in {
            "customer success",
            "solutions engineering"
        }:
            memora_type = "TEAM"

        # TECHNOLOGY
        elif text_lower in {
            "api", "sdk", "gpu", "kms", "s3",
            "cpu", "cli", "hsm", "iam", "rbac",
            "sre", "slo", "ci", "vpc", "qps",
            "ttl", "pii", "sse", "json",
            "jira", "grafana", "slack"
        }:
            memora_type = "TECHNOLOGY"

        # POLICY
        elif text_lower == "sla":
            memora_type = "POLICY"

        # PRODUCT
        elif text_lower == "redwood inference opt":
            memora_type = "PRODUCT"

        # ORGANIZATION
        elif ent.label_ == "ORG":
            memora_type = "ORGANIZATION"

        # PERSON
        elif ent.label_ == "PERSON":
            memora_type = "PERSON"

        # ----------------------------------
        # Ignore anything we couldn't classify
        # ----------------------------------
        if memora_type is None:
            continue

        # ----------------------------------
        # Normalize
        # ----------------------------------
        normalized = " ".join(
            entity_text.lower().split()
        )

        processed.append({
            "original": entity_text,
            "normalized": normalized,
            "memora_type": memora_type,
            "start": ent.start_char,
            "end": ent.end_char
        })

    # --------------------------------------
    # Deduplication
    # --------------------------------------

    unique = []
    seen = set()

    for entity in processed:

        key = (
            entity["normalized"],
            entity["memora_type"]
        )

        if key not in seen:
            seen.add(key)
            unique.append(entity)

    return unique


# ==========================================
# TEST ON FIRST CHUNK
# ==========================================

test_entities = extract_and_process_entities(
    chunks_df.iloc[0]["content"]
)

print("FINAL MEMORA ENTITIES:\n")

for entity in test_entities:
    print(
        entity["original"],
        "->",
        entity["normalized"],
        "->",
        entity["memora_type"]
    )

print("\nTotal entities:", len(test_entities))

FINAL MEMORA ENTITIES:

Customer -> customer -> CUSTOMER
at least 5 -> at least 5 -> TIME
10 business days -> 10 business days -> TIME
at least 72 hours -> at least 72 hours -> TIME
Customer Success -> customer success -> TEAM
Redwood Inference Opt -> redwood inference opt -> PRODUCT
at least 30 days -> at least 30 days -> TIME

Total entities: 7


In [54]:
# ==========================================
# FINAL MEMORA ENTITY EXTRACTION
# ALL CHUNKS
# ==========================================

all_entities = []

for idx, row in chunks_df.iterrows():

    chunk_id = row["chunk_id"]
    text = row["content"]

    entities = extract_and_process_entities(text)

    for entity in entities:

        all_entities.append({
            "chunk_id": chunk_id,
            "entity": entity["original"],
            "normalized_entity": entity["normalized"],
            "entity_type": entity["memora_type"],
            "start": entity["start"],
            "end": entity["end"]
        })


# Create final entity DataFrame
entities_df = pd.DataFrame(all_entities)

print("==========================================")
print("MEMORA ENTITY EXTRACTION COMPLETED")
print("==========================================")

print("Chunks processed:",
      chunks_df["chunk_id"].nunique())

print("Total extracted entities:",
      len(entities_df))

print("\nEntity type distribution:")
print(
    entities_df["entity_type"]
    .value_counts()
)

print("\nUnique normalized entities:")
print(
    entities_df["normalized_entity"]
    .nunique()
)

print("\nSample:")
display(entities_df.head(20))

MEMORA ENTITY EXTRACTION COMPLETED
Chunks processed: 1784
Total extracted entities: 28425

Entity type distribution:
entity_type
ORGANIZATION    8724
TIME            8693
PERSON          7791
TECHNOLOGY      2952
CUSTOMER         111
POLICY           105
TEAM              48
PRODUCT            1
Name: count, dtype: int64

Unique normalized entities:
8930

Sample:


,chunk_id,entity,normalized_entity,entity_type,start,end
0,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,Customer,customer,CUSTOMER,149,157
1,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,at least 5,at least 5,TIME,692,702
2,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,10 business days,10 business days,TIME,826,842
3,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,at least 72 hours,at least 72 hours,TIME,886,903
4,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,Customer Success,customer success,TEAM,1719,1735
5,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,Redwood Inference Opt,redwood inference opt,PRODUCT,2016,2037
6,dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1,at least 30 days,at least 30 days,TIME,2163,2179
7,dsid_9c8cb10d65324c03a8ededbd98c1b058_chunk_1,Redwood Inference,redwood inference,ORGANIZATION,89,106
8,dsid_9c8cb10d65324c03a8ededbd98c1b058_chunk_1,the 72 hours,the 72 hours,TIME,321,333
9,dsid_9c8cb10d65324c03a8ededbd98c1b058_chunk_1,72 hours,72 hours,TIME,354,362


In [55]:
# ==========================================
# MEMORA ENTITY QUALITY - TOP CANDIDATES
# ==========================================

for entity_type in ["ORGANIZATION", "PERSON", "TECHNOLOGY", "TEAM", "POLICY"]:

    print(f"\n{'='*50}")
    print(f"{entity_type}")
    print(f"{'='*50}")

    counts = (
        entities_df[
            entities_df["entity_type"] == entity_type
        ]["normalized_entity"]
        .value_counts()
        .head(50)
    )

    for entity, count in counts.items():
        print(f"{entity:35} {count}")


ORGANIZATION
utc                                 183
doc                                 159
ui                                  121
hosted                              96
eta                                 90
eu                                  79
poc                                 76
cs                                  63
ae                                  55
validate                            48
sso                                 46
dedicated/private                   46
finance                             46
eod                                 45
sec                                 43
adr                                 43
csv                                 43
sku                                 42
p95                                 42
nda                                 41
ip                                  39
token                               39
pdf                                 38
dns                                 37
aws                                 36
dpa     

In [56]:
# MEMORA ENTITY CLASSIFIER - QUALITY CLEANUP

TECHNOLOGY_TERMS = {
    "api", "sre", "slo", "gpu", "kms", "sdk", "ci", "vpc", "pii",
    "json", "ttl", "cli", "rbac", "slack", "qps", "hsm", "s3",
    "iam", "sse", "cpu", "grafana", "jira", "aws", "kubernetes",
    "dns", "tls", "tcp", "csv", "p95", "ip", "token", "tokens/sec"
}

NON_ENTITY_TERMS = {
    "doc", "ui", "hosted", "eta", "eu", "poc", "cs", "ae",
    "validate", "dedicated/private", "finance", "eod", "sec",
    "adr", "sku", "nda", "fallback", "faq", "rollback", "request_id",
    "default", "private", "qa", "rca", "sli", "eng", "post",
    "success", "sup", "arr", "redwood", "redwood inference",
    "redwood private"
}

PERSON_FALSE_POSITIVES = {
    "redwood", "metadata", "config", "dashboard", "trigger",
    "primary", "draft", "drive", "eu-west"
}

# Reclassify obvious technical terms
entities_df.loc[
    entities_df["normalized_entity"].isin(TECHNOLOGY_TERMS),
    "entity_type"
] = "TECHNOLOGY"

# Remove obvious non-entities
entities_df = entities_df[
    ~(
        (entities_df["entity_type"] == "ORGANIZATION") &
        (entities_df["normalized_entity"].isin(NON_ENTITY_TERMS))
    )
].copy()

# Remove obvious PERSON false positives
entities_df = entities_df[
    ~(
        (entities_df["entity_type"] == "PERSON") &
        (entities_df["normalized_entity"].isin(PERSON_FALSE_POSITIVES))
    )
].copy()

print("Cleaned entity count:", len(entities_df))
print("\nEntity type distribution:")
print(entities_df["entity_type"].value_counts())

Cleaned entity count: 26216

Entity type distribution:
entity_type
TIME            8693
PERSON          7118
ORGANIZATION    6853
TECHNOLOGY      3287
CUSTOMER         111
POLICY           105
TEAM              48
PRODUCT            1
Name: count, dtype: int64


In [57]:
# FINAL ENTITY QUALITY CHECK

for entity_type in ["ORGANIZATION", "PERSON", "TECHNOLOGY", "TIME"]:
    print("\n" + "=" * 50)
    print(entity_type)
    print("=" * 50)

    counts = (
        entities_df[entities_df["entity_type"] == entity_type]
        ["normalized_entity"]
        .value_counts()
        .head(40)
    )

    print(counts.to_string())


ORGANIZATION
normalized_entity
utc              183
sso               46
pdf               38
dpa               33
linear            32
soc 2             31
ga                22
phi               22
target            22
hubspot           22
po                21
gcp               21
fyi               21
kpi               21
applied ml        20
gtm               19
cmk               18
terraform         18
cto               18
dek               17
nat               17
ap                17
uuid              17
confluence        17
apac              17
comms             16
smart routing     16
canary            16
change            15
ml                15
node              15
route_id          15
grc               15
iso               14
qbr               14
smb               14
rps               14
nccl              14
nadia rahman      14
scope applies     14

PERSON
normalized_entity
max                 127
sean gallagher       95
markus klein         85
dev patel            62
rafael

In [58]:
# FINAL CONSERVATIVE ENTITY CLEANUP

ORG_FALSE_POSITIVES = {
    "utc", "pdf", "sso", "po", "fyi", "kpi", "cto", "dek",
    "nat", "ap", "uuid", "apac", "comms", "change", "ml",
    "node", "route_id", "scope applies", "grc", "iso",
    "qbr", "smb", "rps", "nccl", "phi", "target"
}

entities_df = entities_df[
    ~(
        (entities_df["entity_type"] == "ORGANIZATION") &
        (entities_df["normalized_entity"].isin(ORG_FALSE_POSITIVES))
    )
].copy()

# Remove duplicate entity mentions while preserving occurrences
entities_df = entities_df.drop_duplicates(
    subset=["chunk_id", "normalized_entity", "entity_type"]
).reset_index(drop=True)

print("FINAL entity count:", len(entities_df))
print("\nFINAL entity type distribution:")
print(entities_df["entity_type"].value_counts())

print("\nTop organizations:")
print(
    entities_df[entities_df["entity_type"] == "ORGANIZATION"]
    ["normalized_entity"]
    .value_counts()
    .head(30)
)


FINAL entity count: 25553

FINAL entity type distribution:
entity_type
TIME            8690
PERSON          7114
ORGANIZATION    6198
TECHNOLOGY      3286
CUSTOMER         111
POLICY           105
TEAM              48
PRODUCT            1
Name: count, dtype: int64

Top organizations:
normalized_entity
dpa                         33
linear                      32
soc 2                       31
ga                          22
hubspot                     22
gcp                         21
applied ml                  20
gtm                         19
cmk                         18
terraform                   18
confluence                  17
canary                      16
smart routing               16
elena                       14
nadia rahman                14
idp                         14
csi                         13
kek                         13
na                          13
mvp                         13
docusign                    13
postgres                    13
acme           

In [59]:
# SAVE FINAL MEMORA ENTITY DATASET

entities_df.to_parquet(
    "MEMORA_entities.parquet",
    index=False
)

entities_df.to_json(
    "MEMORA_entities.jsonl",
    orient="records",
    lines=True
)

print("Saved:")
print("MEMORA_entities.parquet")
print("MEMORA_entities.jsonl")
print("\nRows saved:", len(entities_df))

Saved:
MEMORA_entities.parquet
MEMORA_entities.jsonl

Rows saved: 25553


In [60]:
# Reload EnterpriseRAG-Bench questions

from datasets import load_dataset
import pandas as pd

questions_ds = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions"
)

questions = questions_ds["test"]

questions_df = questions.to_pandas()

print("Benchmark columns:")
print(questions_df.columns.tolist())

print("\nNumber of benchmark questions:", len(questions_df))

display(questions_df.head(3))

Benchmark columns:
['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Number of benchmark questions: 500


,question_id,question_type,source_types,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,[github],What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,[github],What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,[linear],What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...


In [61]:
# MEMORA RETRIEVAL EVALUATION - STEP 2 (CORRECTED)

K_VALUES = [1, 3, 5, 10]

evaluation_results = []

for _, row in questions_df.iterrows():

    query = row["question"]
    expected_docs = set(row["expected_doc_ids"])

    # Retrieve top 10 chunks
    retrieved = retrieve_documents(query, top_k=10)

    result = {
        "question_id": row["question_id"],
        "question": query,
        "expected_docs": expected_docs
    }

    for k in K_VALUES:
        top_k_results = retrieved[:k]

        retrieved_docs = {
            item["doc_id"]
            for item in top_k_results
        }

        result[f"hit@{k}"] = bool(
            retrieved_docs.intersection(expected_docs)
        )

    evaluation_results.append(result)

evaluation_df = pd.DataFrame(evaluation_results)

print("Questions evaluated:", len(evaluation_df))

for k in K_VALUES:
    print(
        f"Hit@{k}: "
        f"{evaluation_df[f'hit@{k}'].mean():.4f}"
    )

Questions evaluated: 500
Hit@1: 0.5500
Hit@3: 0.6560
Hit@5: 0.7160
Hit@10: 0.7840


In [62]:
# RETRIEVAL EVALUATION - FAILURE ANALYSIS

failed_df = evaluation_df[
    evaluation_df["hit@10"] == False
].copy()

print("Total failed questions:", len(failed_df))

print("\nFailure rate:", len(failed_df) / len(evaluation_df))

print("\nFailures by question type:")
print(failed_df["question_id"].count())

display(
    failed_df[
        ["question_id", "question", "expected_docs"]
    ].head(20)
)

Total failed questions: 108

Failure rate: 0.216

Failures by question type:
108


,question_id,question,expected_docs
6,qst_0007,In a rolling investigation of a model regressi...,{dsid_72ec4a9962ba43e88acd61abbba1052d}
7,qst_0008,"In the internal shiproom runner notes, what is...",{dsid_5fc2dba9f6ac4af2b49b4f546a4298d0}
14,qst_0015,What was the high-percentile latency concern r...,{dsid_e75bbf328fe349d2bb39e29b1ed7c9d5}
26,qst_0027,In the Redwood and Acme service assurance nego...,{dsid_214ba4ea822e4d328287194a5e27e632}
31,qst_0032,What performance goal was set for reducing GPU...,{dsid_39e191ec758c41a8adecd223951b8b93}
32,qst_0033,In the security review meeting about private c...,{dsid_05faa1ab6427403888a06d7083c7f0a9}
38,qst_0039,In the 3-hour customer workshop plan for takin...,{dsid_f0567bbb5dbd4a52aef951e11ec13f2b}
40,qst_0041,In the security review call about buying an LL...,{dsid_862bb899faac43feab4f92a12d320e7c}
45,qst_0046,In the Medicore procurement email thread about...,{dsid_3be2b45196414cad9b187551da3ed448}
47,qst_0048,What are the initial weighting percentages for...,{dsid_8412db85d95a45f7bbfb14dacedb4d8b}


In [63]:
# FAILURE ANALYSIS BY QUESTION TYPE

# Add question type from the original benchmark
failed_df = failed_df.merge(
    questions_df[["question_id", "question_type", "source_types"]],
    on="question_id",
    how="left"
)

print("Failures by question type:")
print(
    failed_df["question_type"]
    .value_counts()
)

print("\nFailures by source type:")
print(
    failed_df["source_types"]
    .astype(str)
    .value_counts()
    .head(20)
)

Failures by question type:
question_type
semantic                    42
basic                       28
info_not_found              20
high_level                  10
intra_document_reasoning     6
completeness                 2
Name: count, dtype: int64

Failures by source type:
source_types
[]                  30
['gmail']           15
['google_drive']    14
['hubspot']         11
['confluence']      11
['slack']            8
['fireflies']        7
['linear']           5
['github']           5
['jira']             2
Name: count, dtype: int64


In [64]:
# Separate answerable and info-not-found questions

print("Question type distribution:")
print(questions_df["question_type"].value_counts())

print("\nINFO_NOT_FOUND questions:")
info_not_found = questions_df[
    questions_df["question_type"] == "info_not_found"
]

print("Count:", len(info_not_found))

display(
    info_not_found[
        ["question_id", "question", "expected_doc_ids"]
    ].head(20)
)

Question type distribution:
question_type
basic                       175
semantic                    125
intra_document_reasoning     40
project_related              40
constrained                  30
conflicting_info             20
completeness                 20
miscellaneous                20
info_not_found               20
high_level                   10
Name: count, dtype: int64

INFO_NOT_FOUND questions:
Count: 20


,question_id,question,expected_doc_ids
480,qst_0481,For the hot-route capacity protection rollout ...,[]
481,qst_0482,What exact queue-depth threshold and prefill/d...,[]
482,qst_0483,For the admin activity chronicle's daily Merkl...,[]
483,qst_0484,What is the complete mapping of `billing_tag` ...,[]
484,qst_0485,"In the CSRL ""delayed_with_handover"" admission ...",[]
485,qst_0486,How do we compute and invoice the **per-reques...,[]
486,qst_0487,When a streaming resume attempt fails with HTT...,[]
487,qst_0488,When we use the GPU firmware operator to do a ...,[]
488,qst_0489,In the `observability-pack` status page compon...,[]
489,qst_0490,"For enterprise private deployments, what numer...",[]


In [65]:
# Recalculate retrieval performance on answerable questions only

answerable_df = questions_df[
    questions_df["question_type"] != "info_not_found"
].copy()

print("Total questions:", len(questions_df))
print("Answerable questions:", len(answerable_df))
print("Info-not-found questions:", len(questions_df) - len(answerable_df))

print("\nRetrieval Performance (Answerable Only):")

for k in [1, 3, 5, 10]:
    hits = []

    for _, row in answerable_df.iterrows():
        retrieved = retrieve_documents(row["question"], top_k=k)
        retrieved_doc_ids = {item["doc_id"] for item in retrieved}

        expected_doc_ids = set(row["expected_doc_ids"])

        hit = bool(retrieved_doc_ids & expected_doc_ids)
        hits.append(hit)

    print(f"Hit@{k}: {sum(hits)}/{len(hits)} = {sum(hits)/len(hits):.4f}")

Total questions: 500
Answerable questions: 480
Info-not-found questions: 20

Retrieval Performance (Answerable Only):
Hit@1: 275/480 = 0.5729
Hit@3: 328/480 = 0.6833
Hit@5: 358/480 = 0.7458
Hit@10: 392/480 = 0.8167


In [66]:
# Add question metadata to evaluation results

answerable_eval = evaluation_df.merge(
    questions_df[
        ["question_id", "question_type", "source_types", "expected_doc_ids"]
    ],
    on="question_id",
    how="left"
)

failed_10 = answerable_eval[
    (answerable_eval["question_type"] != "info_not_found") &
    (answerable_eval["hit@10"] == False)
].copy()

print("Genuine Top-10 retrieval failures:", len(failed_10))

display(
    failed_10[
        [
            "question_id",
            "question",
            "question_type",
            "source_types",
            "expected_doc_ids"
        ]
    ].head(20)
)

Genuine Top-10 retrieval failures: 88


,question_id,question,question_type,source_types,expected_doc_ids
6,qst_0007,In a rolling investigation of a model regressi...,basic,[google_drive],[dsid_72ec4a9962ba43e88acd61abbba1052d]
7,qst_0008,"In the internal shiproom runner notes, what is...",basic,[google_drive],[dsid_5fc2dba9f6ac4af2b49b4f546a4298d0]
14,qst_0015,What was the high-percentile latency concern r...,basic,[hubspot],[dsid_e75bbf328fe349d2bb39e29b1ed7c9d5]
26,qst_0027,In the Redwood and Acme service assurance nego...,basic,[gmail],[dsid_214ba4ea822e4d328287194a5e27e632]
31,qst_0032,What performance goal was set for reducing GPU...,basic,[linear],[dsid_39e191ec758c41a8adecd223951b8b93]
32,qst_0033,In the security review meeting about private c...,basic,[fireflies],[dsid_05faa1ab6427403888a06d7083c7f0a9]
38,qst_0039,In the 3-hour customer workshop plan for takin...,basic,[google_drive],[dsid_f0567bbb5dbd4a52aef951e11ec13f2b]
40,qst_0041,In the security review call about buying an LL...,basic,[fireflies],[dsid_862bb899faac43feab4f92a12d320e7c]
45,qst_0046,In the Medicore procurement email thread about...,basic,[gmail],[dsid_3be2b45196414cad9b187551da3ed448]
47,qst_0048,What are the initial weighting percentages for...,basic,[linear],[dsid_8412db85d95a45f7bbfb14dacedb4d8b]


In [67]:
# Failure breakdown by question type

print(
    failed_10["question_type"]
    .value_counts()
    .to_string()
)

print("\nFailure percentage within each question type:")

summary = (
    failed_10["question_type"]
    .value_counts()
    .to_frame("failures")
    .join(
        answerable_eval["question_type"].value_counts()
        .to_frame("total")
    )
)

summary["failure_rate"] = (
    summary["failures"] / summary["total"] * 100
).round(2)

display(summary.sort_values("failure_rate", ascending=False))

question_type
semantic                    42
basic                       28
high_level                  10
intra_document_reasoning     6
completeness                 2

Failure percentage within each question type:


,failures,total,failure_rate
question_type,,,
high_level,10,10,100.0
semantic,42,125,33.6
basic,28,175,16.0
intra_document_reasoning,6,40,15.0
completeness,2,20,10.0


In [68]:
# Inspect all high-level questions and their expected documents

high_level = answerable_eval[
    answerable_eval["question_type"] == "high_level"
].copy()

print("High-level questions:", len(high_level))

display(
    high_level[
        [
            "question_id",
            "question",
            "source_types",
            "expected_doc_ids"
        ]
    ]
)

High-level questions: 10


,question_id,question,source_types,expected_doc_ids
470,qst_0471,What is Redwood Inference's mission statement?,[],[]
471,qst_0472,What is the company's thesis about where compe...,[],[]
472,qst_0473,What security-oriented features are highlighte...,[],[]
473,qst_0474,Which serving-runtime optimizations are explic...,[],[]
474,qst_0475,What policy dimensions can Redwood's smart rou...,[],[]
475,qst_0476,What is the stated differentiation around reli...,[],[]
476,qst_0477,What are the four main revenue streams in Redw...,[],[]
477,qst_0478,What add-on categories are mentioned for Redwo...,[],[]
478,qst_0479,What triggers Redwood's expansion from PLG-led...,[],[]
479,qst_0480,What are the major departments listed in the c...,[],[]


In [69]:
# Final grounded retrieval evaluation
# Exclude question types with no expected documents

grounded_df = questions_df[
    ~questions_df["question_type"].isin(["info_not_found", "high_level"])
].copy()

print("Total benchmark questions:", len(questions_df))
print("Grounded questions:", len(grounded_df))
print("Excluded questions:", len(questions_df) - len(grounded_df))

print("\nGrounded question types:")
print(grounded_df["question_type"].value_counts())

print("\nRetrieval Performance (Grounded Questions Only):")

for k in [1, 3, 5, 10]:
    hits = []

    for _, row in grounded_df.iterrows():
        retrieved = retrieve_documents(row["question"], top_k=k)
        retrieved_doc_ids = {item["doc_id"] for item in retrieved}

        expected_doc_ids = set(row["expected_doc_ids"])

        hits.append(bool(retrieved_doc_ids & expected_doc_ids))

    print(
        f"Hit@{k}: {sum(hits)}/{len(hits)} = "
        f"{sum(hits)/len(hits):.4f}"
    )

Total benchmark questions: 500
Grounded questions: 470
Excluded questions: 30

Grounded question types:
question_type
basic                       175
semantic                    125
intra_document_reasoning     40
project_related              40
constrained                  30
conflicting_info             20
completeness                 20
miscellaneous                20
Name: count, dtype: int64

Retrieval Performance (Grounded Questions Only):
Hit@1: 275/470 = 0.5851
Hit@3: 328/470 = 0.6979
Hit@5: 358/470 = 0.7617
Hit@10: 392/470 = 0.8340


In [70]:
# Analyze genuine retrieval failures on grounded questions

grounded_eval = answerable_eval[
    ~answerable_eval["question_type"].isin(
        ["info_not_found", "high_level"]
    )
].copy()

failed_grounded = grounded_eval[
    grounded_eval["hit@10"] == False
].copy()

print("Grounded Top-10 failures:", len(failed_grounded))

print("\nFailure breakdown:")
print(
    failed_grounded["question_type"]
    .value_counts()
    .to_string()
)

Grounded Top-10 failures: 78

Failure breakdown:
question_type
semantic                    42
basic                       28
intra_document_reasoning     6
completeness                 2


In [71]:
# Inspect the first 15 semantic retrieval failures

semantic_failures = failed_grounded[
    failed_grounded["question_type"] == "semantic"
].copy()

print("Semantic failures:", len(semantic_failures))

display(
    semantic_failures[
        [
            "question_id",
            "question",
            "source_types",
            "expected_doc_ids"
        ]
    ].head(15)
)

Semantic failures: 42


,question_id,question,source_types,expected_doc_ids
181,qst_0182,During a failover between two active locations...,[linear],[dsid_d9ca33382713461aa726403f8ff7dea0]
182,qst_0183,What are the specific gate thresholds used to ...,[confluence],[dsid_693c023c88264795a7e98c7737e6459e]
183,qst_0184,When is the 60 to 90 minute technical deep div...,[gmail],[dsid_74cd6ebda84547e2a534f8f6d1b49739]
184,qst_0185,What was the recommended way to split a year l...,[slack],[dsid_be67d91a95e24fe690cba692cb1762c6]
186,qst_0187,When a production model experiment shows a sus...,[linear],[dsid_a27715e7309c494d812c75cf03bcf916]
189,qst_0190,In the February 2026 update to our inference t...,[confluence],[dsid_b02e6ce0d9814b35af846c309c25e551]
192,qst_0193,For the European customer kickoff scheduled th...,[gmail],[dsid_942ffdd8280a4cb893aca73dcb376ee5]
196,qst_0197,"During a major production outage, what is the ...",[confluence],[dsid_25b0de9bfa854d3c9acfa961555c9d45]
198,qst_0199,After a recent refactor to request grouping an...,[google_drive],[dsid_88f1c9800dd54a75bad05653b7f07a93]
199,qst_0200,For the hospital system that wants to run an i...,[google_drive],[dsid_eab7ef95052c4016aff7a3df422131f0]


In [72]:
# Inspect one semantic retrieval failure in detail

question_id = "qst_0182"

row = grounded_eval[
    grounded_eval["question_id"] == question_id
].iloc[0]

print("QUESTION:")
print(row["question"])

print("\nEXPECTED DOCUMENT:")
print(row["expected_doc_ids"])

print("\nFAISS TOP-10 RESULTS:")

results = retrieve_documents(row["question"], top_k=10)

for r in results:
    print(
        f"\nRank {r['rank']} | Score: {r['score']:.4f}"
        f"\nDoc ID: {r['doc_id']}"
        f"\nSource: {r['source_type']}"
        f"\nTitle: {r['title']}"
        f"\nContent: {r['content'][:500]}"
    )

QUESTION:
During a failover between two active locations, what is the default time window where a short-lived resume credential is accepted on the destination before doing full authentication again?

EXPECTED DOCUMENT:
['dsid_d9ca33382713461aa726403f8ff7dea0']

FAISS TOP-10 RESULTS:

Rank 1 | Score: 0.4483
Doc ID: dsid_b632b2d6f86f41c6820e52ae56549a35
Source: confluence
Title: Severity matrix and escalation triggers (Support P0–P4 ↔ Incident Sev0–Sev3)
Content: a specific SDK version sees truncation; workaround is to adjust client timeouts/reconnect. - Action: Set SUP to **P2**, provide workaround, link known-issue article, escalate only if scope grows. Example 2: Region degradation (latency) - *Scenario:** - eu-west p95 latency 2.5x baseline for 20 minutes; customers report timeouts; partial recovery. - Action: Set SUP tickets to **P1** (or P0 for enterprise hard down), page SRE; move coordination to #incidents if broad. Example 3: Auth outage (SSO / 

Rank 2 | Score: 0.4188
Doc ID: d

In [73]:
# Inspect the expected document for qst_0182

expected_id = "dsid_d9ca33382713461aa726403f8ff7dea0"

expected_chunks = chunks_df[
    chunks_df["doc_id"] == expected_id
]

print("Expected document chunks:", len(expected_chunks))

display(
    expected_chunks[
        ["chunk_id", "title", "content"]
    ].head(10)
)

Expected document chunks: 3


,chunk_id,title,content
1544,dsid_d9ca33382713461aa726403f8ff7dea0_chunk_1,Gateway bifurcation for graceful multi-region ...,description: Problem: During regional disrupti...
1545,dsid_d9ca33382713461aa726403f8ff7dea0_chunk_2,Gateway bifurcation for graceful multi-region ...,fallback to canonical stateless retry path Loa...
1546,dsid_d9ca33382713461aa726403f8ff7dea0_chunk_3,Gateway bifurcation for graceful multi-region ...,- Priya Nair (SRE): Load-shedding rules propos...


In [74]:
# Show the full expected document chunks

for _, r in expected_chunks.iterrows():
    print("\n" + "=" * 100)
    print(f"CHUNK: {r['chunk_id']}")
    print("=" * 100)
    print(r["content"])


CHUNK: dsid_d9ca33382713461aa726403f8ff7dea0_chunk_1
description: Problem: During regional disruptions we currently perform blunt traffic drains or full cutovers that can truncate long-lived streaming sessions and lose in-flight KV state. This causes visible client disconnects, token replays, and SLO violations for tail latency and availability. Goal: Implement a gateway bifurcation pattern that enables graceful takeover between two active regions that share a coordinated per-connection KV checkpointing mechanism, short token-grace window, and circuit-decay policy. The mechanism should: preserve active streaming attachments where possible, reduce user-visible interruptions, maintain SLOs for p95/p99 latency, and allow progressive traffic rebalancing with automated load-shedding triggers and circuit breakers for overloaded endpoints. acceptance_criteria: New gateway bifurcation control plane can execute staged traffic split 100% -> 80/20 -> 50/50 -> 0% with per-connection checkpointing

In [75]:
# Check whether the expected document is retrieved within Top-50

question_id = "qst_0182"

row = grounded_eval[
    grounded_eval["question_id"] == question_id
].iloc[0]

results_50 = retrieve_documents(row["question"], top_k=50)

expected_ids = set(row["expected_doc_ids"])

found = [
    r for r in results_50
    if r["doc_id"] in expected_ids
]

print("Expected document:", expected_ids)

if found:
    for r in found:
        print(
            f"FOUND at rank {r['rank']} | "
            f"score={r['score']:.4f} | "
            f"title={r['title']}"
        )
else:
    print("Expected document NOT found in Top-50")

Expected document: {'dsid_d9ca33382713461aa726403f8ff7dea0'}
Expected document NOT found in Top-50


In [76]:
# Test query expansion on qst_0182

original_query = (
    "During a failover between two active locations, what is the default "
    "time window where a short-lived resume credential is accepted on the "
    "destination before doing full authentication again?"
)

expanded_query = (
    original_query +
    " multi-region failover gateway bifurcation token-grace window "
    "resume token destination region handoff default 45s"
)

print("ORIGINAL QUERY TOP-10")
print("-" * 50)

for r in retrieve_documents(original_query, top_k=10):
    print(f"Rank {r['rank']} | {r['score']:.4f} | {r['title']}")

print("\nEXPANDED QUERY TOP-10")
print("-" * 50)

for r in retrieve_documents(expanded_query, top_k=10):
    print(f"Rank {r['rank']} | {r['score']:.4f} | {r['title']}")

ORIGINAL QUERY TOP-10
--------------------------------------------------
Rank 1 | 0.4483 | Severity matrix and escalation triggers (Support P0–P4 ↔ Incident Sev0–Sev3)
Rank 2 | 0.4188 | Partial-result Checkpointing and Client Resume Contract
Rank 3 | 0.4137 | introduce-server-timebox-and-idempotent-cancel-fallbacks-for-streaming-compat
Rank 4 | 0.4023 | streaming-session-snapshot-client-ack-add-resume-token-openai-compat
Rank 5 | 0.3989 | GPU Warm-Pool Cross-Account Handoff Runbook (2025)
Rank 6 | 0.3936 | introduce-session-decorator-to-simplify-cross-sdk-streams-and-credential-rotation
Rank 7 | 0.3868 | introduce-server-timebox-and-idempotent-cancel-fallbacks-for-streaming-compat
Rank 8 | 0.3828 | Partial-result Checkpointing and Client Resume Contract
Rank 9 | 0.3780 | Production secret rotation runbook
Rank 10 | 0.3606 | Resilient streaming schema: fallback stubs and idempotent tool-relay semantics

EXPANDED QUERY TOP-10
--------------------------------------------------
Rank 1 | 0.

In [77]:
# Compare failed semantic questions with their current top FAISS result

sample_failures = semantic_failures.head(10)

for _, row in sample_failures.iterrows():
    results = retrieve_documents(row["question"], top_k=1)

    print("\n" + "=" * 100)
    print("QUESTION:", row["question"])
    print("EXPECTED:", row["expected_doc_ids"][0])

    if results:
        r = results[0]
        print("TOP RESULT:", r["title"])
        print("SCORE:", round(r["score"], 4))
        print("TOP RESULT DOC:", r["doc_id"])


QUESTION: During a failover between two active locations, what is the default time window where a short-lived resume credential is accepted on the destination before doing full authentication again?
EXPECTED: dsid_d9ca33382713461aa726403f8ff7dea0
TOP RESULT: Severity matrix and escalation triggers (Support P0–P4 ↔ Incident Sev0–Sev3)
SCORE: 0.4483
TOP RESULT DOC: dsid_b632b2d6f86f41c6820e52ae56549a35

QUESTION: What are the specific gate thresholds used to automatically decide whether a compressed model variant is allowed, canaried, or blocked, including the limits for chat similarity drop, code pass rate change, retrieval embedding quality, and acceptable latency and cost changes?
EXPECTED: dsid_693c023c88264795a7e98c7737e6459e
TOP RESULT: Quantization Profile Compatibility & Fallback Standard (Route-level Enablement)
SCORE: 0.4686
TOP RESULT DOC: dsid_a6efecc786c9436499eb000b8489ae52

QUESTION: When is the 60 to 90 minute technical deep dive scheduled with the healthcare client abou

In [78]:
# Simple query expansion: emphasize important terms from the question

import re

def expand_query(question):
    # Remove common question words
    stopwords = {
        "what", "was", "were", "when", "where", "who", "how",
        "is", "are", "the", "a", "an", "of", "to", "for", "in",
        "on", "with", "and", "or", "from", "during", "after",
        "before", "about", "our", "their", "they", "this", "that",
        "what", "did", "does", "do", "can", "should"
    }

    words = re.findall(r'\b[a-zA-Z0-9_-]+\b', question.lower())

    keywords = [
        w for w in words
        if w not in stopwords and len(w) > 2
    ]

    # Keep unique terms while preserving order
    keywords = list(dict.fromkeys(keywords))

    return question + " " + " ".join(keywords)


# Test on the first 10 semantic failures
for _, row in sample_failures.iterrows():
    expanded = expand_query(row["question"])
    results = retrieve_documents(expanded, top_k=1)

    print("\n" + "=" * 100)
    print("QUESTION:", row["question"])
    print("EXPANDED:", expanded)

    if results:
        r = results[0]
        print("TOP RESULT:", r["title"])
        print("SCORE:", round(r["score"], 4))
        print("TOP RESULT DOC:", r["doc_id"])


QUESTION: During a failover between two active locations, what is the default time window where a short-lived resume credential is accepted on the destination before doing full authentication again?
EXPANDED: During a failover between two active locations, what is the default time window where a short-lived resume credential is accepted on the destination before doing full authentication again? failover between two active locations default time window short-lived resume credential accepted destination doing full authentication again
TOP RESULT: Severity matrix and escalation triggers (Support P0–P4 ↔ Incident Sev0–Sev3)
SCORE: 0.4334
TOP RESULT DOC: dsid_b632b2d6f86f41c6820e52ae56549a35

QUESTION: What are the specific gate thresholds used to automatically decide whether a compressed model variant is allowed, canaried, or blocked, including the limits for chat similarity drop, code pass rate change, retrieval embedding quality, and acceptable latency and cost changes?
EXPANDED: What a

In [79]:
# Find where the correct document appears in the FULL FAISS ranking
# for all semantic failures

semantic_rank_analysis = []

for _, row in semantic_failures.iterrows():
    results = retrieve_documents(row["question"], top_k=len(chunks_df))

    expected_doc = row["expected_doc_ids"][0]

    rank = None
    for r in results:
        if r["doc_id"] == expected_doc:
            rank = r["rank"]
            break

    semantic_rank_analysis.append({
        "question_id": row["question_id"],
        "rank": rank,
        "question": row["question"]
    })

semantic_rank_df = pd.DataFrame(semantic_rank_analysis)

print("Semantic failures:", len(semantic_rank_df))
print("Correct document found:", semantic_rank_df["rank"].notna().sum())
print("Correct document NOT found:", semantic_rank_df["rank"].isna().sum())

print("\nRank distribution:")
print(
    semantic_rank_df["rank"]
    .dropna()
    .describe()
)

print("\nRank buckets:")
print(
    pd.cut(
        semantic_rank_df["rank"],
        bins=[0, 10, 20, 50, 100, 250, 500, len(chunks_df)],
        labels=["1-10", "11-20", "21-50", "51-100", "101-250", "251-500", "501+"],
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nExamples:")
print(
    semantic_rank_df
    .sort_values("rank", na_position="last")
    [["question_id", "rank", "question"]]
    .head(15)
    .to_string(index=False)
)

Semantic failures: 42
Correct document found: 42
Correct document NOT found: 0

Rank distribution:
count      42.000000
mean       96.428571
std       202.106968
min        13.000000
25%        23.000000
50%        32.500000
75%        95.500000
max      1273.000000
Name: rank, dtype: float64

Rank buckets:
rank
1-10        0
11-20       8
21-50      18
51-100      6
101-250     7
251-500     2
501+        1
Name: count, dtype: int64

Examples:
question_id  rank                                                                                                                                                                                                                                                                                                             question
   qst_0216    13                                                                                                                    In our cross team outage process, what is the expected update rhythm for the shared inciden

In [80]:
import importlib.util

print("sentence_transformers:", importlib.util.find_spec("sentence_transformers") is not None)
print("transformers:", importlib.util.find_spec("transformers") is not None)
print("torch:", importlib.util.find_spec("torch") is not None)

sentence_transformers: True
transformers: True
torch: True


In [81]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded successfully!


In [82]:
# Rerank only cases where the correct document is already in FAISS Top-50

rerank_results = []

for _, row in semantic_failures.iterrows():

    candidates = retrieve_documents(row["question"], top_k=50)
    expected_doc = row["expected_doc_ids"][0]

    # Skip if correct document is outside the Top-50 candidate set
    matching = [r for r in candidates if r["doc_id"] == expected_doc]

    if not matching:
        continue

    pairs = [
        [row["question"], candidate["content"]]
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    new_rank = next(
        rank for rank, candidate in enumerate(reranked, start=1)
        if candidate["doc_id"] == expected_doc
    )

    original_rank = matching[0]["rank"]

    rerank_results.append({
        "question_id": row["question_id"],
        "original_rank": original_rank,
        "reranked_rank": new_rank
    })

rerank_df = pd.DataFrame(rerank_results)

print("Cases eligible for Top-50 reranking:", len(rerank_df), "/ 42")

print("Improved:",
      (rerank_df["reranked_rank"] < rerank_df["original_rank"]).sum())

print("Unchanged:",
      (rerank_df["reranked_rank"] == rerank_df["original_rank"]).sum())

print("Worsened:",
      (rerank_df["reranked_rank"] > rerank_df["original_rank"]).sum())

print("\nTop-5 after reranking:",
      (rerank_df["reranked_rank"] <= 5).sum(),
      "/", len(rerank_df))

print("Top-10 after reranking:",
      (rerank_df["reranked_rank"] <= 10).sum(),
      "/", len(rerank_df))

print("\nRank comparison:")
print(
    rerank_df
    .sort_values("reranked_rank")
    .to_string(index=False)
)

Cases eligible for Top-50 reranking: 26 / 42
Improved: 25
Unchanged: 0
Worsened: 1

Top-5 after reranking: 16 / 26
Top-10 after reranking: 20 / 26

Rank comparison:
question_id  original_rank  reranked_rank
   qst_0213             29              1
   qst_0216             13              1
   qst_0237             21              1
   qst_0281             31              1
   qst_0274             40              1
   qst_0277             23              1
   qst_0261             16              1
   qst_0251             23              1
   qst_0294             13              1
   qst_0193             36              2
   qst_0250             17              2
   qst_0282             49              3
   qst_0185             16              3
   qst_0234             34              3
   qst_0247             28              3
   qst_0260             29              4
   qst_0288             27              6
   qst_0183             20              6
   qst_0243             14           

In [83]:
# MEMORA: FAISS + Cross-Encoder Reranked Retrieval

def retrieve_reranked(query, candidate_k=50, top_k=5):
    """
    Retrieve candidate chunks using FAISS, then rerank them
    using the Cross-Encoder.

    candidate_k = number of FAISS candidates
    top_k       = final number of chunks returned
    """

    # Step 1: FAISS candidate retrieval
    candidates = retrieve_documents(query, top_k=candidate_k)

    if not candidates:
        return []

    # Step 2: Prepare question-document pairs
    pairs = [
        [query, candidate["content"]]
        for candidate in candidates
    ]

    # Step 3: Cross-Encoder scoring
    scores = reranker.predict(pairs)

    # Step 4: Attach reranking scores
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    # Step 5: Sort by Cross-Encoder score
    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Step 6: Return final Top-K
    return reranked[:top_k]


print("Reranked retrieval function created successfully!")

Reranked retrieval function created successfully!


In [84]:
# Test the new reranked retrieval on a known semantic failure

test_query = semantic_failures.iloc[0]["question"]

results = retrieve_reranked(
    test_query,
    candidate_k=50,
    top_k=10
)

print("QUESTION:")
print(test_query)

print("\n" + "=" * 100)

for r in results:
    print(
        f"Rank {r['rank']} | "
        f"FAISS score: {r['score']:.4f} | "
        f"Rerank score: {r['rerank_score']:.4f}"
    )
    print("Title:", r["title"])
    print("Doc ID:", r["doc_id"])
    print()

QUESTION:
During a failover between two active locations, what is the default time window where a short-lived resume credential is accepted on the destination before doing full authentication again?

Rank 3 | FAISS score: 0.4137 | Rerank score: -2.3102
Title: introduce-server-timebox-and-idempotent-cancel-fallbacks-for-streaming-compat
Doc ID: dsid_9550250a59e74f1bbd5612480b2e7100

Rank 9 | FAISS score: 0.3780 | Rerank score: -2.6825
Title: Production secret rotation runbook
Doc ID: dsid_cf06e75be4f94156ac338f62e3ee028b

Rank 2 | FAISS score: 0.4188 | Rerank score: -2.6845
Title: Partial-result Checkpointing and Client Resume Contract
Doc ID: dsid_91d7f359508c4476a6f717ea2b33aecc

Rank 16 | FAISS score: 0.3476 | Rerank score: -4.4851
Title: Emission Stability and Tool-Orchestration: Soft-Fail & Streaming Acceptance
Doc ID: dsid_f94b93a4bc7948ddab45c371187dc6c6

Rank 7 | FAISS score: 0.3868 | Rerank score: -4.5282
Title: introduce-server-timebox-and-idempotent-cancel-fallbacks-for-strea

In [85]:
# Verify reranking on a case where the correct document is in FAISS Top-50

test_row = semantic_failures[
    semantic_failures["question_id"] == "qst_0213"
].iloc[0]

results = retrieve_reranked(
    test_row["question"],
    candidate_k=50,
    top_k=5
)

print("QUESTION:")
print(test_row["question"])

print("\nEXPECTED DOC:")
print(test_row["expected_doc_ids"][0])

print("\nRERANKED RESULTS:")
for new_rank, r in enumerate(results, start=1):
    print(
        f"\nNew Rank: {new_rank}"
        f"\nOriginal FAISS Rank: {r['rank']}"
        f"\nFAISS Score: {r['score']:.4f}"
        f"\nRerank Score: {r['rerank_score']:.4f}"
        f"\nTitle: {r['title']}"
        f"\nDoc ID: {r['doc_id']}"
    )

QUESTION:
For a content templating company planning tens of millions of very short generations during a Q3 ramp, what was the requested plan for a time limited evaluation that included a large free usage allowance?

EXPECTED DOC:
dsid_ad14c3cb06c4494093dfcfff5098d89d

RERANKED RESULTS:

New Rank: 1
Original FAISS Rank: 29
FAISS Score: 0.4109
Rerank Score: -3.3018
Title: BrambleBox Content Labs
Doc ID: dsid_ad14c3cb06c4494093dfcfff5098d89d

New Rank: 2
Original FAISS Rank: 44
FAISS Score: 0.3993
Rerank Score: -8.6722
Title: Cleanup stale test instances and grant access for infra-eco team (archived volumes)
Doc ID: dsid_6df52fdb96ae4edcb76464738bca3340

New Rank: 3
Original FAISS Rank: 33
FAISS Score: 0.4092
Rerank Score: -9.0001
Title: DossierHQ: procure ephemeral collaborator pool (120 seats) + SCIM/SSO mapping and billing tags
Doc ID: dsid_1d5a3cde61104276b218c05dc101a546

New Rank: 4
Original FAISS Rank: 23
FAISS Score: 0.4150
Rerank Score: -9.4694
Title: Dedicated capacity options a

In [86]:
# MEMORA: Full grounded benchmark evaluation
# FAISS Top-50 + Cross-Encoder Reranking

rerank_eval = []

for i, (_, row) in enumerate(grounded_eval.iterrows(), start=1):

    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # FAISS retrieves 50 candidates
    candidates = retrieve_documents(question, top_k=50)

    # Cross-Encoder reranking
    pairs = [
        [question, candidate["content"]]
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Remove duplicate documents while preserving reranked order
    unique_docs = []
    seen_docs = set()

    for candidate in reranked:
        if candidate["doc_id"] not in seen_docs:
            unique_docs.append(candidate)
            seen_docs.add(candidate["doc_id"])

    # Calculate reranked hits
    retrieved_docs = [r["doc_id"] for r in unique_docs]

    rerank_eval.append({
        "question_id": row["question_id"],
        "hit@1": any(
            doc in expected_docs
            for doc in retrieved_docs[:1]
        ),
        "hit@3": any(
            doc in expected_docs
            for doc in retrieved_docs[:3]
        ),
        "hit@5": any(
            doc in expected_docs
            for doc in retrieved_docs[:5]
        ),
        "hit@10": any(
            doc in expected_docs
            for doc in retrieved_docs[:10]
        )
    })

    if i % 25 == 0:
        print(f"Processed {i}/{len(grounded_eval)} questions...")

rerank_evaluation_df = pd.DataFrame(rerank_eval)

print("\n" + "=" * 60)
print("MEMORA RERANKED RETRIEVAL RESULTS")
print("=" * 60)

print(
    "Hit@1 :", round(rerank_evaluation_df["hit@1"].mean() * 100, 2), "%"
)
print(
    "Hit@3 :", round(rerank_evaluation_df["hit@3"].mean() * 100, 2), "%"
)
print(
    "Hit@5 :", round(rerank_evaluation_df["hit@5"].mean() * 100, 2), "%"
)
print(
    "Hit@10:", round(rerank_evaluation_df["hit@10"].mean() * 100, 2), "%"
)

Processed 25/470 questions...
Processed 50/470 questions...
Processed 75/470 questions...
Processed 100/470 questions...
Processed 125/470 questions...
Processed 150/470 questions...
Processed 175/470 questions...
Processed 200/470 questions...
Processed 225/470 questions...
Processed 250/470 questions...
Processed 275/470 questions...
Processed 300/470 questions...
Processed 325/470 questions...
Processed 350/470 questions...
Processed 375/470 questions...
Processed 400/470 questions...
Processed 425/470 questions...
Processed 450/470 questions...

MEMORA RERANKED RETRIEVAL RESULTS
Hit@1 : 77.66 %
Hit@3 : 85.11 %
Hit@5 : 88.51 %
Hit@10: 91.06 %


In [87]:
# Save the reranked evaluation results

rerank_evaluation_df.to_parquet(
    "MEMORA_reranked_evaluation.parquet",
    index=False
)

rerank_evaluation_df.to_json(
    "MEMORA_reranked_evaluation.jsonl",
    orient="records",
    lines=True
)

print("Saved:")
print("MEMORA_reranked_evaluation.parquet")
print("MEMORA_reranked_evaluation.jsonl")

Saved:
MEMORA_reranked_evaluation.parquet
MEMORA_reranked_evaluation.jsonl


In [88]:
# Analyze remaining retrieval failures after reranking

rerank_analysis = rerank_evaluation_df.merge(
    grounded_eval[
        ["question_id", "question_type", "question", "expected_doc_ids"]
    ],
    on="question_id",
    how="left"
)

remaining_failures = rerank_analysis[
    rerank_analysis["hit@10"] == False
].copy()

print("Total grounded questions:", len(rerank_analysis))
print("Remaining Top-10 failures:", len(remaining_failures))
print(
    "Remaining failure rate:",
    round(len(remaining_failures) / len(rerank_analysis) * 100, 2),
    "%"
)

print("\nFailures by question type:")
print(
    remaining_failures["question_type"]
    .value_counts()
    .to_string()
)

print("\nRemaining failures:")
print(
    remaining_failures[
        ["question_id", "question_type", "question"]
    ].to_string(index=False)
)

Total grounded questions: 470
Remaining Top-10 failures: 42
Remaining failure rate: 8.94 %

Failures by question type:
question_type
semantic                    25
basic                       12
intra_document_reasoning     3
conflicting_info             1
completeness                 1

Remaining failures:
question_id            question_type                                                                                                                                                                                                                                                                                                                                                                                                                     question
   qst_0008                    basic                                                                                                                                                                                                             

In [89]:
# Test whether increasing the FAISS candidate pool from Top-50 to Top-100
# helps recover the remaining 42 failures.

rerank100_results = []

for _, row in remaining_failures.iterrows():

    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # Retrieve a larger candidate pool
    candidates = retrieve_documents(question, top_k=100)

    # Rerank all 100 candidates
    pairs = [
        [question, candidate["content"]]
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Deduplicate documents
    unique_docs = []
    seen_docs = set()

    for candidate in reranked:
        if candidate["doc_id"] not in seen_docs:
            unique_docs.append(candidate)
            seen_docs.add(candidate["doc_id"])

    retrieved_docs = [r["doc_id"] for r in unique_docs]

    rerank100_results.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "hit@1": any(doc in expected_docs for doc in retrieved_docs[:1]),
        "hit@3": any(doc in expected_docs for doc in retrieved_docs[:3]),
        "hit@5": any(doc in expected_docs for doc in retrieved_docs[:5]),
        "hit@10": any(doc in expected_docs for doc in retrieved_docs[:10])
    })

rerank100_df = pd.DataFrame(rerank100_results)

print("Remaining failures tested:", len(rerank100_df))

print("\nRecovery with FAISS Top-100 + Reranker:")
print("Hit@1 :", round(rerank100_df["hit@1"].mean() * 100, 2), "%")
print("Hit@3 :", round(rerank100_df["hit@3"].mean() * 100, 2), "%")
print("Hit@5 :", round(rerank100_df["hit@5"].mean() * 100, 2), "%")
print("Hit@10:", round(rerank100_df["hit@10"].mean() * 100, 2), "%")

Remaining failures tested: 42

Recovery with FAISS Top-100 + Reranker:
Hit@1 : 23.81 %
Hit@3 : 23.81 %
Hit@5 : 28.57 %
Hit@10: 30.95 %


In [90]:
# Diagnose whether the expected document is present in FAISS Top-100
# for the remaining 42 failures.

diagnosis = []

for _, row in remaining_failures.iterrows():

    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    candidates = retrieve_documents(question, top_k=100)

    candidate_doc_ids = [c["doc_id"] for c in candidates]

    found = [doc for doc in expected_docs if doc in candidate_doc_ids]

    if found:
        # Original FAISS rank of the first expected document
        faiss_rank = min(
            candidate_doc_ids.index(doc) + 1
            for doc in found
        )
        status = "Type B: In Top-100"
    else:
        faiss_rank = None
        status = "Type A: Not in Top-100"

    diagnosis.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "faiss_rank": faiss_rank,
        "status": status
    })

diagnosis_df = pd.DataFrame(diagnosis)

print(diagnosis_df["status"].value_counts())

print("\nBreakdown:")
print(
    diagnosis_df
    .groupby(["status", "question_type"])
    .size()
    .sort_values(ascending=False)
)

print("\nExamples of Type A:")
print(
    diagnosis_df[
        diagnosis_df["status"] == "Type A: Not in Top-100"
    ].head(10)
)

status
Type B: In Top-100        25
Type A: Not in Top-100    17
Name: count, dtype: int64

Breakdown:
status                  question_type           
Type B: In Top-100      semantic                    15
Type A: Not in Top-100  semantic                    10
Type B: In Top-100      basic                        7
Type A: Not in Top-100  basic                        5
Type B: In Top-100      intra_document_reasoning     2
Type A: Not in Top-100  completeness                 1
                        intra_document_reasoning     1
Type B: In Top-100      conflicting_info             1
dtype: int64

Examples of Type A:
   question_id question_type  faiss_rank                  status
0     qst_0008         basic         NaN  Type A: Not in Top-100
1     qst_0015         basic         NaN  Type A: Not in Top-100
8     qst_0120         basic         NaN  Type A: Not in Top-100
9     qst_0144         basic         NaN  Type A: Not in Top-100
11    qst_0173         basic         NaN  Type A:

In [91]:
# Inspect Type-B failures: expected document is in Top-100
# but was not recovered into the final Top-10.

type_b = diagnosis_df[
    diagnosis_df["status"] == "Type B: In Top-100"
].copy()

print("Type-B failures:", len(type_b))

print("\nFAISS rank statistics:")
print(type_b["faiss_rank"].describe())

print("\nType-B failures sorted by FAISS rank:")
print(
    type_b[
        ["question_id", "question_type", "faiss_rank"]
    ].sort_values("faiss_rank").to_string(index=False)
)

Type-B failures: 25

FAISS rank statistics:
count    25.000000
mean     42.960000
std      30.177917
min       1.000000
25%      21.000000
50%      50.000000
75%      59.000000
max      97.000000
Name: faiss_rank, dtype: float64

Type-B failures sorted by FAISS rank:
question_id            question_type  faiss_rank
   qst_0220                 semantic         1.0
   qst_0256                 semantic         1.0
   qst_0414         conflicting_info         2.0
   qst_0226                 semantic         9.0
   qst_0310 intra_document_reasoning        10.0
   qst_0190                 semantic        15.0
   qst_0206                 semantic        21.0
   qst_0184                 semantic        23.0
   qst_0231                 semantic        23.0
   qst_0267                 semantic        25.0
   qst_0246                 semantic        25.0
   qst_0116                    basic        44.0
   qst_0150                    basic        50.0
   qst_0064                    basic        51

In [92]:
# Compare FAISS rank vs Cross-Encoder reranked rank
# for the 25 Type-B failures.

type_b_comparison = []

for _, row in type_b.iterrows():

    question_id = row["question_id"]

    # Get original question and expected document
    original_row = remaining_failures[
        remaining_failures["question_id"] == question_id
    ].iloc[0]

    question = original_row["question"]
    expected_docs = original_row["expected_doc_ids"]

    # Retrieve Top-100
    candidates = retrieve_documents(question, top_k=100)

    # Rerank
    pairs = [[question, c["content"]] for c in candidates]
    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Find expected document positions
    faiss_doc_ids = [c["doc_id"] for c in candidates]
    reranked_doc_ids = [c["doc_id"] for c in reranked]

    expected_doc = next(
        (doc for doc in expected_docs if doc in faiss_doc_ids),
        None
    )

    faiss_rank = faiss_doc_ids.index(expected_doc) + 1
    rerank_rank = reranked_doc_ids.index(expected_doc) + 1

    expected_candidate = next(
        c for c in reranked
        if c["doc_id"] == expected_doc
    )

    type_b_comparison.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "faiss_rank": faiss_rank,
        "rerank_rank": rerank_rank,
        "rerank_score": expected_candidate["rerank_score"]
    })

type_b_comparison_df = pd.DataFrame(type_b_comparison)

print(
    type_b_comparison_df
    .sort_values("rerank_rank")
    .to_string(index=False)
)

question_id            question_type  faiss_rank  rerank_rank  rerank_score
   qst_0027                    basic          80            1      2.892065
   qst_0046                    basic          97            1      2.845868
   qst_0064                    basic          51            1      0.589891
   qst_0069                    basic          54            1      3.074650
   qst_0182                 semantic          64            1      1.397983
   qst_0212                 semantic          91            1      1.247588
   qst_0283                 semantic          52            1     -3.255955
   qst_0272                 semantic          74            1     -0.264844
   qst_0311 intra_document_reasoning          55            1      1.006061
   qst_0256                 semantic           1            4     -0.292501
   qst_0231                 semantic          23            5     -6.075386
   qst_0094                    basic          51           10     -3.352046
   qst_0150 

In [93]:
# Test hybrid FAISS + Cross-Encoder scoring on the 42 remaining failures.

weights = [0.9, 0.8, 0.7, 0.6, 0.5]

hybrid_results = []

for _, row in remaining_failures.iterrows():

    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    candidates = retrieve_documents(question, top_k=100)

    pairs = [[question, c["content"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)

    faiss_scores = [c["score"] for c in candidates]

    # Min-max normalize both scores
    def minmax(scores):
        scores = np.array(scores, dtype=float)
        if scores.max() == scores.min():
            return np.zeros_like(scores)
        return (scores - scores.min()) / (scores.max() - scores.min())

    faiss_norm = minmax(faiss_scores)
    rerank_norm = minmax(rerank_scores)

    for i, candidate in enumerate(candidates):
        candidate["faiss_norm"] = faiss_norm[i]
        candidate["rerank_norm"] = rerank_norm[i]

    for faiss_weight in weights:

        rerank_weight = 1 - faiss_weight

        ranked = sorted(
            candidates,
            key=lambda x:
                faiss_weight * x["faiss_norm"]
                + rerank_weight * x["rerank_norm"],
            reverse=True
        )

        # Deduplicate documents
        unique_docs = []
        seen_docs = set()

        for candidate in ranked:
            if candidate["doc_id"] not in seen_docs:
                unique_docs.append(candidate)
                seen_docs.add(candidate["doc_id"])

        retrieved_docs = [c["doc_id"] for c in unique_docs]

        hybrid_results.append({
            "question_id": row["question_id"],
            "faiss_weight": faiss_weight,
            "hit@1": any(d in expected_docs for d in retrieved_docs[:1]),
            "hit@3": any(d in expected_docs for d in retrieved_docs[:3]),
            "hit@5": any(d in expected_docs for d in retrieved_docs[:5]),
            "hit@10": any(d in expected_docs for d in retrieved_docs[:10])
        })

hybrid_df = pd.DataFrame(hybrid_results)

print("Hybrid recovery on remaining 42 failures:\n")

print(
    hybrid_df
    .groupby("faiss_weight")[["hit@1", "hit@3", "hit@5", "hit@10"]]
    .mean()
    .mul(100)
    .round(2)
)

Hybrid recovery on remaining 42 failures:

              hit@1  hit@3  hit@5  hit@10
faiss_weight                             
0.5            0.00   9.52  14.29   23.81
0.6            2.38   7.14  11.90   19.05
0.7            2.38   7.14   9.52   11.90
0.8            2.38   7.14   7.14   11.90
0.9            2.38   7.14   7.14    9.52


In [94]:
!pip -q install rank_bm25

from rank_bm25 import BM25Okapi
import re

# Prepare chunk corpus for lexical retrieval
corpus = chunks_df["content"].fillna("").astype(str).tolist()

tokenized_corpus = [
    re.findall(r"\b\w+\b", text.lower())
    for text in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")
print("Documents:", len(corpus))

BM25 ready.
Documents: 1785


In [95]:
# Test BM25 retrieval on the 17 Type-A failures.

type_a = diagnosis_df[
    diagnosis_df["status"] == "Type A: Not in Top-100"
]

bm25_results = []

for _, row in type_a.iterrows():

    question_id = row["question_id"]

    original_row = remaining_failures[
        remaining_failures["question_id"] == question_id
    ].iloc[0]

    question = original_row["question"]
    expected_docs = original_row["expected_doc_ids"]

    # BM25 retrieval
    query_tokens = re.findall(r"\b\w+\b", question.lower())
    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:100]

    retrieved_docs = [
        chunks_df.iloc[idx]["doc_id"]
        for idx in top_indices
    ]

    # First occurrence of expected document
    found_ranks = [
        i + 1
        for i, doc in enumerate(retrieved_docs)
        if doc in expected_docs
    ]

    bm25_rank = min(found_ranks) if found_ranks else None

    bm25_results.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "bm25_rank": bm25_rank,
        "found_in_top100": bm25_rank is not None
    })

bm25_df = pd.DataFrame(bm25_results)

print("Type-A failures tested:", len(bm25_df))

print("\nBM25 recovery:")
print(
    bm25_df["found_in_top100"]
    .value_counts()
)

print("\nBM25 rank statistics:")
print(
    bm25_df[bm25_df["found_in_top100"]]["bm25_rank"]
    .describe()
)

print("\nResults:")
print(
    bm25_df
    .sort_values(
        "bm25_rank",
        na_position="last"
    )
    .to_string(index=False)
)

Type-A failures tested: 17

BM25 recovery:
found_in_top100
True    17
Name: count, dtype: int64

BM25 rank statistics:
count    17.000000
mean      7.294118
std      10.970214
min       1.000000
25%       1.000000
50%       1.000000
75%       9.000000
max      34.000000
Name: bm25_rank, dtype: float64

Results:
question_id            question_type  bm25_rank  found_in_top100
   qst_0008                    basic          1             True
   qst_0144                    basic          1             True
   qst_0199                 semantic          1             True
   qst_0187                 semantic          1             True
   qst_0200                 semantic          1             True
   qst_0317 intra_document_reasoning          1             True
   qst_0280                 semantic          1             True
   qst_0208                 semantic          1             True
   qst_0291                 semantic          1             True
   qst_0293                 semantic 

In [96]:
# Analyze BM25 recovery on the 17 Type-A failures

print("===== BM25 TYPE-A RECOVERY ANALYSIS =====")

total = len(bm25_df)
recovered = bm25_df["found_in_top100"].sum()
missed = total - recovered

print(f"Total Type-A failures: {total}")
print(f"Recovered by BM25:    {recovered}")
print(f"Still missed:        {missed}")

if recovered > 0:
    print(f"Recovery rate:        {recovered / total * 100:.2f}%")

    print("\n===== RECOVERED QUESTIONS =====")
    print(
        bm25_df[bm25_df["found_in_top100"]]
        .sort_values("bm25_rank")
        .to_string(index=False)
    )

print("\n===== STILL MISSED =====")
print(
    bm25_df[~bm25_df["found_in_top100"]]
    .to_string(index=False)
)

print("\n===== RANK STATISTICS =====")

if recovered > 0:
    print(
        bm25_df[bm25_df["found_in_top100"]]["bm25_rank"]
        .describe()
    )
else:
    print("No Type-A failures were recovered by BM25.")

===== BM25 TYPE-A RECOVERY ANALYSIS =====
Total Type-A failures: 17
Recovered by BM25:    17
Still missed:        0
Recovery rate:        100.00%

===== RECOVERED QUESTIONS =====
question_id            question_type  bm25_rank  found_in_top100
   qst_0008                    basic          1             True
   qst_0144                    basic          1             True
   qst_0199                 semantic          1             True
   qst_0187                 semantic          1             True
   qst_0200                 semantic          1             True
   qst_0317 intra_document_reasoning          1             True
   qst_0280                 semantic          1             True
   qst_0208                 semantic          1             True
   qst_0291                 semantic          1             True
   qst_0293                 semantic          2             True
   qst_0015                    basic          2             True
   qst_0239                 semantic     

In [97]:
# Compare FAISS and BM25 candidate coverage on all 42 remaining failures

coverage_results = []

for _, row in remaining_failures.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # -------------------------
    # FAISS Top-100
    # -------------------------
    faiss_candidates = retrieve_documents(question, top_k=100)

    faiss_docs = [
        item["doc_id"]
        for item in faiss_candidates
    ]

    faiss_rank = next(
        (
            i + 1
            for i, doc in enumerate(faiss_docs)
            if doc in expected_docs
        ),
        None
    )

    # -------------------------
    # BM25 Top-100
    # -------------------------
    query_tokens = re.findall(r"\b\w+\b", question.lower())
    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = np.argsort(bm25_scores)[::-1][:100]

    bm25_docs = [
        chunks_df.iloc[idx]["doc_id"]
        for idx in bm25_indices
    ]

    bm25_rank = next(
        (
            i + 1
            for i, doc in enumerate(bm25_docs)
            if doc in expected_docs
        ),
        None
    )

    # -------------------------
    # Combined candidate pool
    # -------------------------
    combined_docs = list(dict.fromkeys(
        faiss_docs + bm25_docs
    ))

    combined_rank = next(
        (
            i + 1
            for i, doc in enumerate(combined_docs)
            if doc in expected_docs
        ),
        None
    )

    coverage_results.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "faiss_rank": faiss_rank,
        "bm25_rank": bm25_rank,
        "combined_rank": combined_rank,
        "faiss_found": faiss_rank is not None,
        "bm25_found": bm25_rank is not None,
        "combined_found": combined_rank is not None
    })

coverage_df = pd.DataFrame(coverage_results)

print("===== CANDIDATE COVERAGE =====")

print("Total remaining failures:", len(coverage_df))

print("\nFAISS Top-100:")
print(coverage_df["faiss_found"].value_counts())

print("\nBM25 Top-100:")
print(coverage_df["bm25_found"].value_counts())

print("\nCombined FAISS + BM25:")
print(coverage_df["combined_found"].value_counts())

print("\n===== RECOVERED BY COMBINATION =====")

print(
    coverage_df[
        (~coverage_df["faiss_found"]) &
        (coverage_df["bm25_found"])
    ]
    .sort_values("bm25_rank")
    .to_string(index=False)
)

print("\n===== COMBINED MISSES =====")

print(
    coverage_df[
        ~coverage_df["combined_found"]
    ]
    .to_string(index=False)
)

===== CANDIDATE COVERAGE =====
Total remaining failures: 42

FAISS Top-100:
faiss_found
True     25
False    17
Name: count, dtype: int64

BM25 Top-100:
bm25_found
True     41
False     1
Name: count, dtype: int64

Combined FAISS + BM25:
combined_found
True    42
Name: count, dtype: int64

===== RECOVERED BY COMBINATION =====
question_id            question_type  faiss_rank  bm25_rank  combined_rank  faiss_found  bm25_found  combined_found
   qst_0008                    basic         NaN        1.0             70        False        True            True
   qst_0144                    basic         NaN        1.0             84        False        True            True
   qst_0199                 semantic         NaN        1.0             85        False        True            True
   qst_0187                 semantic         NaN        1.0             80        False        True            True
   qst_0200                 semantic         NaN        1.0             84        False     

In [98]:
# ============================================
# HYBRID FAISS + BM25 + CROSS-ENCODER
# Test on the 42 remaining failures
# ============================================

hybrid_results = []

for _, row in remaining_failures.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # ----------------------------------------
    # 1. FAISS Top-100 candidates
    # ----------------------------------------
    faiss_candidates = retrieve_documents(
        question,
        top_k=100
    )

    # ----------------------------------------
    # 2. BM25 Top-100 candidates
    # ----------------------------------------
    query_tokens = re.findall(
        r"\b\w+\b",
        question.lower()
    )

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = np.argsort(
        bm25_scores
    )[::-1][:100]

    bm25_candidates = []

    for idx in bm25_indices:
        bm25_candidates.append({
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "doc_id": chunks_df.iloc[idx]["doc_id"],
            "source_type": chunks_df.iloc[idx]["source_type"],
            "title": chunks_df.iloc[idx]["title"],
            "content": chunks_df.iloc[idx]["content"]
        })

    # ----------------------------------------
    # 3. Combine candidates
    # Deduplicate by chunk_id
    # ----------------------------------------
    candidate_dict = {}

    for candidate in faiss_candidates:
        candidate_dict[candidate["chunk_id"]] = candidate

    for candidate in bm25_candidates:
        if candidate["chunk_id"] not in candidate_dict:
            candidate_dict[candidate["chunk_id"]] = candidate

    combined_candidates = list(candidate_dict.values())

    # ----------------------------------------
    # 4. Cross-Encoder reranking
    # ----------------------------------------
    pairs = [
        [question, candidate["content"]]
        for candidate in combined_candidates
    ]

    rerank_scores = reranker.predict(pairs)

    for candidate, score in zip(
        combined_candidates,
        rerank_scores
    ):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        combined_candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # ----------------------------------------
    # 5. Find expected document rank
    # ----------------------------------------
    expected_rank = next(
        (
            i + 1
            for i, candidate in enumerate(reranked)
            if candidate["doc_id"] in expected_docs
        ),
        None
    )

    hybrid_results.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "hybrid_rank": expected_rank,
        "hybrid_hit@1": expected_rank is not None and expected_rank <= 1,
        "hybrid_hit@3": expected_rank is not None and expected_rank <= 3,
        "hybrid_hit@5": expected_rank is not None and expected_rank <= 5,
        "hybrid_hit@10": expected_rank is not None and expected_rank <= 10,
        "candidate_count": len(combined_candidates)
    })


# ============================================
# RESULTS
# ============================================

hybrid_df = pd.DataFrame(hybrid_results)

print("======================================")
print("HYBRID RETRIEVAL RESULTS")
print("======================================")

print("Total failures tested:", len(hybrid_df))

print(
    "Hit@1:",
    hybrid_df["hybrid_hit@1"].sum(),
    "/",
    len(hybrid_df)
)

print(
    "Hit@3:",
    hybrid_df["hybrid_hit@3"].sum(),
    "/",
    len(hybrid_df)
)

print(
    "Hit@5:",
    hybrid_df["hybrid_hit@5"].sum(),
    "/",
    len(hybrid_df)
)

print(
    "Hit@10:",
    hybrid_df["hybrid_hit@10"].sum(),
    "/",
    len(hybrid_df)
)

print("\nPercentage:")

print(
    "Hit@1:",
    round(hybrid_df["hybrid_hit@1"].mean() * 100, 2),
    "%"
)

print(
    "Hit@3:",
    round(hybrid_df["hybrid_hit@3"].mean() * 100, 2),
    "%"
)

print(
    "Hit@5:",
    round(hybrid_df["hybrid_hit@5"].mean() * 100, 2),
    "%"
)

print(
    "Hit@10:",
    round(hybrid_df["hybrid_hit@10"].mean() * 100, 2),
    "%"
)

print("\n===== HYBRID MISSES =====")

print(
    hybrid_df[
        ~hybrid_df["hybrid_hit@10"]
    ].to_string(index=False)
)

HYBRID RETRIEVAL RESULTS
Total failures tested: 42
Hit@1: 19 / 42
Hit@3: 22 / 42
Hit@5: 25 / 42
Hit@10: 28 / 42

Percentage:
Hit@1: 45.24 %
Hit@3: 52.38 %
Hit@5: 59.52 %
Hit@10: 66.67 %

===== HYBRID MISSES =====
question_id question_type  hybrid_rank  hybrid_hit@1  hybrid_hit@3  hybrid_hit@5  hybrid_hit@10  candidate_count
   qst_0094         basic           12         False         False         False          False              184
   qst_0144         basic           19         False         False         False          False              176
   qst_0150         basic           13         False         False         False          False              176
   qst_0187      semantic           26         False         False         False          False              179
   qst_0190      semantic           44         False         False         False          False              189
   qst_0199      semantic           17         False         False         False          False              

In [99]:
# ============================================
# CONTROLLED HYBRID RETRIEVAL
# FAISS Top-50 + BM25 Top-50
# Then Cross-Encoder reranking
# ============================================

controlled_hybrid_results = []

for _, row in remaining_failures.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # ----------------------------------------
    # 1. FAISS Top-50
    # ----------------------------------------
    faiss_candidates = retrieve_documents(
        question,
        top_k=50
    )

    # ----------------------------------------
    # 2. BM25 Top-50
    # ----------------------------------------
    query_tokens = re.findall(
        r"\b\w+\b",
        question.lower()
    )

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = np.argsort(
        bm25_scores
    )[::-1][:50]

    bm25_candidates = []

    for idx in bm25_indices:
        bm25_candidates.append({
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "doc_id": chunks_df.iloc[idx]["doc_id"],
            "source_type": chunks_df.iloc[idx]["source_type"],
            "title": chunks_df.iloc[idx]["title"],
            "content": chunks_df.iloc[idx]["content"]
        })

    # ----------------------------------------
    # 3. Combine + deduplicate by chunk_id
    # ----------------------------------------
    candidate_dict = {}

    for candidate in faiss_candidates:
        candidate_dict[candidate["chunk_id"]] = candidate

    for candidate in bm25_candidates:
        if candidate["chunk_id"] not in candidate_dict:
            candidate_dict[candidate["chunk_id"]] = candidate

    combined_candidates = list(candidate_dict.values())

    # ----------------------------------------
    # 4. Cross-Encoder reranking
    # ----------------------------------------
    pairs = [
        [question, candidate["content"]]
        for candidate in combined_candidates
    ]

    rerank_scores = reranker.predict(pairs)

    for candidate, score in zip(
        combined_candidates,
        rerank_scores
    ):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        combined_candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # ----------------------------------------
    # 5. Find expected document rank
    # ----------------------------------------
    expected_rank = next(
        (
            i + 1
            for i, candidate in enumerate(reranked)
            if candidate["doc_id"] in expected_docs
        ),
        None
    )

    controlled_hybrid_results.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "hybrid_rank": expected_rank,
        "hit@1": expected_rank is not None and expected_rank <= 1,
        "hit@3": expected_rank is not None and expected_rank <= 3,
        "hit@5": expected_rank is not None and expected_rank <= 5,
        "hit@10": expected_rank is not None and expected_rank <= 10,
        "candidate_count": len(combined_candidates)
    })


# ============================================
# RESULTS
# ============================================

controlled_hybrid_df = pd.DataFrame(
    controlled_hybrid_results
)

print("======================================")
print("CONTROLLED HYBRID RESULTS")
print("======================================")

print(
    "Total failures tested:",
    len(controlled_hybrid_df)
)

for k in [1, 3, 5, 10]:

    hits = controlled_hybrid_df[f"hit@{k}"].sum()
    percentage = (
        controlled_hybrid_df[f"hit@{k}"].mean() * 100
    )

    print(
        f"Hit@{k}: {hits} / 42 "
        f"({percentage:.2f}%)"
    )

print("\nAverage candidate count:",
      round(controlled_hybrid_df["candidate_count"].mean(), 2))

print("\n===== CONTROLLED HYBRID MISSES =====")

print(
    controlled_hybrid_df[
        ~controlled_hybrid_df["hit@10"]
    ].to_string(index=False)
)

CONTROLLED HYBRID RESULTS
Total failures tested: 42
Hit@1: 19 / 42 (45.24%)
Hit@3: 24 / 42 (57.14%)
Hit@5: 25 / 42 (59.52%)
Hit@10: 28 / 42 (66.67%)

Average candidate count: 92.48

===== CONTROLLED HYBRID MISSES =====
question_id question_type  hybrid_rank  hit@1  hit@3  hit@5  hit@10  candidate_count
   qst_0116         basic         15.0  False  False  False   False               92
   qst_0144         basic         18.0  False  False  False   False               91
   qst_0150         basic         13.0  False  False  False   False               89
   qst_0187      semantic         28.0  False  False  False   False               90
   qst_0190      semantic         23.0  False  False  False   False               96
   qst_0199      semantic         12.0  False  False  False   False               94
   qst_0206      semantic         32.0  False  False  False   False               91
   qst_0220      semantic         59.0  False  False  False   False               78
   qst_0226     

In [100]:
# ============================================
# CONTROLLED HYBRID RETRIEVAL
# FAISS Top-50 + BM25 Top-50
# Then Cross-Encoder reranking
# ============================================

controlled_hybrid_results = []

for _, row in remaining_failures.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # ----------------------------------------
    # 1. FAISS Top-50
    # ----------------------------------------
    faiss_candidates = retrieve_documents(
        question,
        top_k=50
    )

    # ----------------------------------------
    # 2. BM25 Top-50
    # ----------------------------------------
    query_tokens = re.findall(
        r"\b\w+\b",
        question.lower()
    )

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = np.argsort(
        bm25_scores
    )[::-1][:50]

    bm25_candidates = []

    for idx in bm25_indices:
        bm25_candidates.append({
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "doc_id": chunks_df.iloc[idx]["doc_id"],
            "source_type": chunks_df.iloc[idx]["source_type"],
            "title": chunks_df.iloc[idx]["title"],
            "content": chunks_df.iloc[idx]["content"]
        })

    # ----------------------------------------
    # 3. Combine + deduplicate by chunk_id
    # ----------------------------------------
    candidate_dict = {}

    for candidate in faiss_candidates:
        candidate_dict[candidate["chunk_id"]] = candidate

    for candidate in bm25_candidates:
        if candidate["chunk_id"] not in candidate_dict:
            candidate_dict[candidate["chunk_id"]] = candidate

    combined_candidates = list(candidate_dict.values())

    # ----------------------------------------
    # 4. Cross-Encoder reranking
    # ----------------------------------------
    pairs = [
        [question, candidate["content"]]
        for candidate in combined_candidates
    ]

    rerank_scores = reranker.predict(pairs)

    for candidate, score in zip(
        combined_candidates,
        rerank_scores
    ):
        candidate["rerank_score"] = float(score)

    reranked = sorted(
        combined_candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # ----------------------------------------
    # 5. Find expected document rank
    # ----------------------------------------
    expected_rank = next(
        (
            i + 1
            for i, candidate in enumerate(reranked)
            if candidate["doc_id"] in expected_docs
        ),
        None
    )

    controlled_hybrid_results.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "hybrid_rank": expected_rank,
        "hit@1": expected_rank is not None and expected_rank <= 1,
        "hit@3": expected_rank is not None and expected_rank <= 3,
        "hit@5": expected_rank is not None and expected_rank <= 5,
        "hit@10": expected_rank is not None and expected_rank <= 10,
        "candidate_count": len(combined_candidates)
    })


# ============================================
# RESULTS
# ============================================

controlled_hybrid_df = pd.DataFrame(
    controlled_hybrid_results
)

print("======================================")
print("CONTROLLED HYBRID RESULTS")
print("======================================")

print(
    "Total failures tested:",
    len(controlled_hybrid_df)
)

for k in [1, 3, 5, 10]:

    hits = controlled_hybrid_df[f"hit@{k}"].sum()
    percentage = (
        controlled_hybrid_df[f"hit@{k}"].mean() * 100
    )

    print(
        f"Hit@{k}: {hits} / 42 "
        f"({percentage:.2f}%)"
    )

print("\nAverage candidate count:",
      round(controlled_hybrid_df["candidate_count"].mean(), 2))

print("\n===== CONTROLLED HYBRID MISSES =====")

print(
    controlled_hybrid_df[
        ~controlled_hybrid_df["hit@10"]
    ].to_string(index=False)
)

CONTROLLED HYBRID RESULTS
Total failures tested: 42
Hit@1: 19 / 42 (45.24%)
Hit@3: 24 / 42 (57.14%)
Hit@5: 25 / 42 (59.52%)
Hit@10: 28 / 42 (66.67%)

Average candidate count: 92.48

===== CONTROLLED HYBRID MISSES =====
question_id question_type  hybrid_rank  hit@1  hit@3  hit@5  hit@10  candidate_count
   qst_0116         basic         15.0  False  False  False   False               92
   qst_0144         basic         18.0  False  False  False   False               91
   qst_0150         basic         13.0  False  False  False   False               89
   qst_0187      semantic         28.0  False  False  False   False               90
   qst_0190      semantic         23.0  False  False  False   False               96
   qst_0199      semantic         12.0  False  False  False   False               94
   qst_0206      semantic         32.0  False  False  False   False               91
   qst_0220      semantic         59.0  False  False  False   False               78
   qst_0226     

In [101]:
# ============================================
# RRF + CROSS-ENCODER
# Test on 42 remaining failures
# ============================================

rrf_results = []

# RRF constant
RRF_K = 60

for _, row in remaining_failures.iterrows():

    question_id = row["question_id"]
    question = row["question"]
    expected_docs = row["expected_doc_ids"]

    # ----------------------------------------
    # 1. FAISS Top-50
    # ----------------------------------------
    faiss_candidates = retrieve_documents(
        question,
        top_k=50
    )

    # ----------------------------------------
    # 2. BM25 Top-50
    # ----------------------------------------
    query_tokens = re.findall(
        r"\b\w+\b",
        question.lower()
    )

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = np.argsort(
        bm25_scores
    )[::-1][:50]

    bm25_candidates = []

    for idx in bm25_indices:
        bm25_candidates.append({
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "doc_id": chunks_df.iloc[idx]["doc_id"],
            "source_type": chunks_df.iloc[idx]["source_type"],
            "title": chunks_df.iloc[idx]["title"],
            "content": chunks_df.iloc[idx]["content"]
        })

    # ----------------------------------------
    # 3. RRF score
    # ----------------------------------------
    rrf_scores = {}

    candidate_dict = {}

    for rank, candidate in enumerate(
        faiss_candidates, start=1
    ):
        chunk_id = candidate["chunk_id"]

        candidate_dict[chunk_id] = candidate

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0)
            + 1 / (RRF_K + rank)
        )

    for rank, candidate in enumerate(
        bm25_candidates, start=1
    ):
        chunk_id = candidate["chunk_id"]

        if chunk_id not in candidate_dict:
            candidate_dict[chunk_id] = candidate

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0)
            + 1 / (RRF_K + rank)
        )

    # ----------------------------------------
    # 4. Sort using RRF
    # ----------------------------------------
    rrf_candidates = sorted(
        candidate_dict.values(),
        key=lambda x: rrf_scores[x["chunk_id"]],
        reverse=True
    )

    # ----------------------------------------
    # 5. Cross-Encoder reranking
    # ----------------------------------------
    pairs = [
        [question, candidate["content"]]
        for candidate in rrf_candidates
    ]

    rerank_scores = reranker.predict(pairs)

    for candidate, score in zip(
        rrf_candidates,
        rerank_scores
    ):
        candidate["rerank_score"] = float(score)

    final_candidates = sorted(
        rrf_candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # ----------------------------------------
    # 6. Expected document rank
    # ----------------------------------------
    expected_rank = next(
        (
            i + 1
            for i, candidate in enumerate(final_candidates)
            if candidate["doc_id"] in expected_docs
        ),
        None
    )

    rrf_results.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "rrf_rank": expected_rank,
        "hit@1": expected_rank is not None and expected_rank <= 1,
        "hit@3": expected_rank is not None and expected_rank <= 3,
        "hit@5": expected_rank is not None and expected_rank <= 5,
        "hit@10": expected_rank is not None and expected_rank <= 10,
        "candidate_count": len(rrf_candidates)
    })


# ============================================
# RESULTS
# ============================================

rrf_df = pd.DataFrame(rrf_results)

print("======================================")
print("RRF + CROSS-ENCODER RESULTS")
print("======================================")

print(
    "Total failures tested:",
    len(rrf_df)
)

for k in [1, 3, 5, 10]:

    hits = rrf_df[f"hit@{k}"].sum()
    percentage = (
        rrf_df[f"hit@{k}"].mean() * 100
    )

    print(
        f"Hit@{k}: {hits} / 42 "
        f"({percentage:.2f}%)"
    )

print(
    "\nAverage candidate count:",
    round(
        rrf_df["candidate_count"].mean(),
        2
    )
)

print("\n===== RRF MISSES =====")

print(
    rrf_df[
        ~rrf_df["hit@10"]
    ].to_string(index=False)
)

RRF + CROSS-ENCODER RESULTS
Total failures tested: 42
Hit@1: 19 / 42 (45.24%)
Hit@3: 24 / 42 (57.14%)
Hit@5: 25 / 42 (59.52%)
Hit@10: 28 / 42 (66.67%)

Average candidate count: 92.48

===== RRF MISSES =====
question_id question_type  rrf_rank  hit@1  hit@3  hit@5  hit@10  candidate_count
   qst_0116         basic      15.0  False  False  False   False               92
   qst_0144         basic      18.0  False  False  False   False               91
   qst_0150         basic      13.0  False  False  False   False               89
   qst_0187      semantic      28.0  False  False  False   False               90
   qst_0190      semantic      23.0  False  False  False   False               96
   qst_0199      semantic      12.0  False  False  False   False               94
   qst_0206      semantic      32.0  False  False  False   False               91
   qst_0220      semantic      59.0  False  False  False   False               78
   qst_0226      semantic      16.0  False  False  Fals

In [102]:
# ============================================
# MEMORA — CONTEXT BUILDER
# Uses existing FAISS + Cross-Encoder retrieval
# ============================================

def build_rag_context(
    query,
    candidate_k=50,
    top_k=5,
    max_chars_per_chunk=3000
):
    """
    Retrieve and prepare grounded context for MEMORA RAG.

    Pipeline:
        Query
          ↓
        FAISS Top-50
          ↓
        Cross-Encoder
          ↓
        Top-K
          ↓
        Context Builder
    """

    # ----------------------------------------
    # 1. Retrieve candidates
    # ----------------------------------------
    candidates = retrieve_documents(
        query,
        top_k=candidate_k
    )

    if not candidates:
        return {
            "query": query,
            "context": "",
            "sources": [],
            "num_sources": 0
        }

    # ----------------------------------------
    # 2. Cross-Encoder reranking
    # ----------------------------------------
    pairs = [
        [query, item["content"]]
        for item in candidates
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # ----------------------------------------
    # 3. Select final evidence
    # ----------------------------------------
    selected = reranked[:top_k]

    # ----------------------------------------
    # 4. Build grounded context
    # ----------------------------------------
    context_parts = []
    sources = []

    for i, item in enumerate(selected, start=1):

        content = str(
            item.get("content", "")
        ).strip()

        # Prevent extremely large context
        content = content[:max_chars_per_chunk]

        source_block = f"""
[Evidence {i}]
Document ID: {item.get("doc_id", "N/A")}
Chunk ID: {item.get("chunk_id", "N/A")}
Title: {item.get("title", "N/A")}
Source Type: {item.get("source_type", "N/A")}
Semantic/Rerank Score: {item.get("rerank_score", 0.0):.4f}

Content:
{content}
"""

        context_parts.append(
            source_block.strip()
        )

        sources.append({
            "evidence_id": i,
            "doc_id": item.get("doc_id"),
            "chunk_id": item.get("chunk_id"),
            "title": item.get("title"),
            "source_type": item.get("source_type"),
            "rerank_score": item.get("rerank_score"),
            "content": content
        })

    context = "\n\n".join(context_parts)

    return {
        "query": query,
        "context": context,
        "sources": sources,
        "num_sources": len(sources)
    }


# ============================================
# TEST
# ============================================

test_query = (
    "What are the requirements for customer "
    "communication during a model rollout?"
)

rag_context = build_rag_context(
    test_query,
    candidate_k=50,
    top_k=5
)

print("======================================")
print("MEMORA RAG CONTEXT")
print("======================================")

print("Query:")
print(rag_context["query"])

print("\nNumber of sources:")
print(rag_context["num_sources"])

print("\n===== SOURCES =====")

for source in rag_context["sources"]:

    print(
        f"\n[{source['evidence_id']}] "
        f"{source['title']}"
    )

    print(
        "Document:",
        source["doc_id"]
    )

    print(
        "Chunk:",
        source["chunk_id"]
    )

    print(
        "Rerank score:",
        round(source["rerank_score"], 4)
    )

    print(
        "Content preview:",
        source["content"][:500].replace("\n", " ")
    )

print("\n===== FULL CONTEXT =====")
print(rag_context["context"])

MEMORA RAG CONTEXT
Query:
What are the requirements for customer communication during a model rollout?

Number of sources:
5

===== SOURCES =====

[1] Support Process: Customer Communications Requirements for Model Rollouts
Document: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c
Chunk: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1
Rerank score: 7.1184
Content preview: Purpose This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules. Who must be notified Always notify - Any Dedicated customer receiving the new version as default - Any Private customer (VPC/on-prem) impacted by the change Notify when material behavior changes For Hosted API customers, notify when any of the following change materially for a commonly used a

[2] Status Page and Incident Comms: Guidance for Rollout-Related Issues
Document: dsid_2dab968306594149a14ef27b2a1641d1
Chunk: dsid_2da

In [103]:
# ============================================
# MEMORA — GROUNDED RAG PROMPT BUILDER
# ============================================

def build_rag_prompt(rag_context):
    """
    Creates a strict grounded RAG prompt from
    the Context Builder output.
    """

    query = rag_context["query"]
    context = rag_context["context"]

    prompt = f"""
You are MEMORA, an Enterprise Organizational
Memory and Intelligence assistant.

Your job is to answer the user's question using
ONLY the evidence retrieved from the MEMORA knowledge base.

========================
USER QUESTION
========================

{query}

========================
RETRIEVED EVIDENCE
========================

{context}

========================
GROUNDING RULES
========================

1. Answer only using the retrieved evidence.

2. Do NOT use outside knowledge to invent facts.

3. Do NOT invent people, projects, organizations,
   technologies, decisions, dates, policies, or relationships.

4. Every important factual statement should be supported
   by one or more evidence references.

5. Cite evidence using:
   [Evidence 1]
   [Evidence 2]
   etc.

6. If multiple sources support the same statement,
   cite all relevant sources.

7. If the evidence contains conflicting information,
   explicitly mention the conflict and identify the
   supporting sources.

8. If the retrieved evidence is insufficient to answer
   the question, clearly state:

   "I could not find sufficient evidence in MEMORA's
   indexed knowledge to answer this question."

9. Do not guess or fill missing information.

10. Prefer concise, factual answers.

11. Preserve important numbers, dates, requirements,
    conditions, and exceptions exactly as supported
    by the evidence.

========================
ANSWER FORMAT
========================

Answer:
<direct answer>

Evidence:
- [Evidence X] <why this evidence supports the answer>
- [Evidence Y] <why this evidence supports the answer>

Confidence:
<High / Medium / Low>

If the evidence is insufficient, explain what
information is missing.

========================
FINAL ANSWER
========================
"""

    return prompt


# ============================================
# TEST
# ============================================

rag_prompt = build_rag_prompt(
    rag_context
)

print("======================================")
print("MEMORA GROUNDED RAG PROMPT")
print("======================================")

print(rag_prompt)

MEMORA GROUNDED RAG PROMPT

You are MEMORA, an Enterprise Organizational
Memory and Intelligence assistant.

Your job is to answer the user's question using
ONLY the evidence retrieved from the MEMORA knowledge base.

USER QUESTION

What are the requirements for customer communication during a model rollout?

RETRIEVED EVIDENCE

[Evidence 1]
Document ID: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c
Chunk ID: dsid_2ce9ae40d5d347c1add2b31c4d5a8c3c_chunk_1
Title: Support Process: Customer Communications Requirements for Model Rollouts
Source Type: confluence
Semantic/Rerank Score: 7.1184

Content:
Purpose This page defines the minimum customer-facing communication requirements for production model version rollouts. Rollout plans MUST include a Customer communications section that follows these rules. Who must be notified Always notify - Any Dedicated customer receiving the new version as default - Any Private customer (VPC/on-prem) impacted by the change Notify when material behavior changes For

In [104]:
# ============================================
# MEMORA — RAG ANSWER GENERATOR
# ============================================

def generate_rag_answer(rag_prompt, llm_callable):
    """
    Sends the grounded MEMORA prompt to an LLM.

    Parameters
    ----------
    rag_prompt : str
        Prompt created by build_rag_prompt()

    llm_callable : function
        Function that accepts a prompt and returns
        the generated text.

    Returns
    -------
    str
        Grounded LLM response.
    """

    if not rag_prompt or not rag_prompt.strip():
        raise ValueError("RAG prompt cannot be empty.")

    if llm_callable is None:
        raise ValueError(
            "No LLM provider has been configured."
        )

    response = llm_callable(rag_prompt)

    if response is None:
        raise RuntimeError(
            "LLM returned no response."
        )

    response = str(response).strip()

    if not response:
        raise RuntimeError(
            "LLM returned an empty response."
        )

    return response


# ============================================
# DEMO / MOCK LLM
# ============================================

def demo_llm(prompt):
    """
    Local deterministic demo provider.

    This is NOT a real LLM.
    It is used only to verify that the
    RAG pipeline is connected correctly.
    """

    return """
Answer:

For a production model version rollout, customer
communications must identify what is changing, why it
is changing, when the change will occur, the expected
impact, the opt-out or version-pinning process, support
contacts, and the rollback posture. [Evidence 1]

Notification timing depends on the customer type:
Dedicated customers should generally be notified at least
5 business days before the change window, Private customers
follow their contract with a default of 10 business days,
and Hosted customers require at least 72 hours' notice for
material behavior changes. [Evidence 1]

Customer communications must also receive the required
approvals, including Customer Success and the Product
Manager for inference. Regulated Private customers also
require Security review. [Evidence 1][Evidence 3]

If a rollout causes qualifying customer impact or a rollback
under the specified conditions, status-page communication
may also be required. [Evidence 2]

Evidence:
- [Evidence 1] Main customer communication requirements.
- [Evidence 2] Rollout-related incident/status-page rules.
- [Evidence 3] Private deployment requirements.

Confidence:
High
"""


# ============================================
# TEST THE COMPLETE RAG GENERATION FLOW
# ============================================

try:

    generated_answer = generate_rag_answer(
        rag_prompt,
        demo_llm
    )

    print("======================================")
    print("MEMORA RAG ANSWER")
    print("======================================")

    print(generated_answer)

except Exception as e:

    print("RAG generation error:")
    print(type(e).__name__, ":", str(e))

MEMORA RAG ANSWER
Answer:

For a production model version rollout, customer
communications must identify what is changing, why it
is changing, when the change will occur, the expected
impact, the opt-out or version-pinning process, support
contacts, and the rollback posture. [Evidence 1]

Notification timing depends on the customer type:
Dedicated customers should generally be notified at least
5 business days before the change window, Private customers
follow their contract with a default of 10 business days,
and Hosted customers require at least 72 hours' notice for
material behavior changes. [Evidence 1]

Customer communications must also receive the required
approvals, including Customer Success and the Product
Manager for inference. Regulated Private customers also
require Security review. [Evidence 1][Evidence 3]

If a rollout causes qualifying customer impact or a rollback
under the specified conditions, status-page communication
may also be required. [Evidence 2]

Evidence:
- [

In [105]:
# ============================================
# MEMORA — REAL LLM PROVIDER
# ============================================

# Install OpenAI SDK if needed
!pip -q install openai

import os
from openai import OpenAI


# ============================================
# CONFIGURATION
# ============================================

# Set these in Colab before using the real LLM:
#
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
# os.environ["MEMORA_LLM_MODEL"] = "gpt-4o-mini"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MEMORA_LLM_MODEL = os.getenv(
    "MEMORA_LLM_MODEL",
    "gpt-4o-mini"
)


# ============================================
# OPENAI LLM FUNCTION
# ============================================

def openai_llm(prompt):
    """
    Generate a response using an OpenAI model.

    The model receives the complete grounded
    MEMORA prompt produced by build_rag_prompt().
    """

    if not OPENAI_API_KEY:
        raise ValueError(
            "OPENAI_API_KEY is not configured. "
            "Set it in the environment first."
        )

    client = OpenAI(
        api_key=OPENAI_API_KEY
    )

    response = client.chat.completions.create(
        model=MEMORA_LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are MEMORA, an enterprise "
                    "organizational memory assistant. "
                    "Follow the supplied grounding rules "
                    "strictly."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1
    )

    answer = response.choices[0].message.content

    if not answer:
        raise RuntimeError(
            "The LLM returned an empty response."
        )

    return answer.strip()


# ============================================
# GENERIC LLM PROVIDER
# ============================================

def memora_llm(prompt):
    """
    Main MEMORA LLM interface.

    The rest of the project should call this function
    instead of calling an LLM provider directly.
    """

    provider = os.getenv(
        "MEMORA_LLM_PROVIDER",
        "openai"
    ).lower()

    if provider == "openai":
        return openai_llm(prompt)

    elif provider == "demo":
        return demo_llm(prompt)

    else:
        raise ValueError(
            f"Unsupported MEMORA_LLM_PROVIDER: {provider}"
        )


# ============================================
# PROVIDER STATUS
# ============================================

print("======================================")
print("MEMORA LLM PROVIDER")
print("======================================")

print(
    "Provider:",
    os.getenv("MEMORA_LLM_PROVIDER", "openai")
)

print(
    "Model:",
    MEMORA_LLM_MODEL
)

print(
    "API key configured:",
    bool(OPENAI_API_KEY)
)

MEMORA LLM PROVIDER
Provider: openai
Model: gpt-4o-mini
API key configured: False


In [106]:
# ============================================
# MEMORA — COMPLETE QUERY PIPELINE
# ============================================

def ask_memora(
    query,
    candidate_k=50,
    top_k=5
):
    """
    Complete MEMORA question-answering pipeline.

    Flow:

    User Query
        ↓
    FAISS Retrieval
        ↓
    Cross-Encoder Reranking
        ↓
    Context Builder
        ↓
    Grounded RAG Prompt
        ↓
    LLM
        ↓
    Answer + Evidence
    """

    if not query or not query.strip():
        raise ValueError(
            "Query cannot be empty."
        )

    query = query.strip()

    # ----------------------------------------
    # STEP 1 — Retrieve + build context
    # ----------------------------------------

    rag_context = build_rag_context(
        query=query,
        candidate_k=candidate_k,
        top_k=top_k
    )

    # ----------------------------------------
    # STEP 2 — Check retrieval
    # ----------------------------------------

    if rag_context["num_sources"] == 0:

        return {
            "query": query,
            "answer": (
                "I could not find sufficient evidence "
                "in MEMORA's indexed knowledge."
            ),
            "sources": [],
            "num_sources": 0,
            "retrieval_strategy": "faiss_cross_encoder"
        }

    # ----------------------------------------
    # STEP 3 — Build grounded prompt
    # ----------------------------------------

    prompt = build_rag_prompt(
        rag_context
    )

    # ----------------------------------------
    # STEP 4 — Generate answer
    # ----------------------------------------

    answer = memora_llm(
        prompt
    )

    # ----------------------------------------
    # STEP 5 — Return structured response
    # ----------------------------------------

    return {
        "query": query,
        "answer": answer,
        "sources": rag_context["sources"],
        "num_sources": rag_context["num_sources"],
        "retrieval_strategy": "faiss_cross_encoder",
        "candidate_k": candidate_k,
        "top_k": top_k
    }


# ============================================
# DEMO MODE TEST
# ============================================

# Use demo provider so an API key is NOT required.

os.environ["MEMORA_LLM_PROVIDER"] = "demo"

test_query = (
    "What are the requirements for customer "
    "communication during a model rollout?"
)

result = ask_memora(
    test_query,
    candidate_k=50,
    top_k=5
)


# ============================================
# DISPLAY RESULT
# ============================================

print("======================================")
print("MEMORA")
print("======================================")

print("\nQUESTION:")
print(result["query"])

print("\nANSWER:")
print(result["answer"])

print("\n======================================")
print("RETRIEVAL INFORMATION")
print("======================================")

print(
    "Strategy:",
    result["retrieval_strategy"]
)

print(
    "Candidates:",
    result["candidate_k"]
)

print(
    "Final sources:",
    result["num_sources"]
)

print("\n======================================")
print("EVIDENCE")
print("======================================")

for source in result["sources"]:

    print(
        f"\n[{source['evidence_id']}] "
        f"{source['title']}"
    )

    print(
        "Document:",
        source["doc_id"]
    )

    print(
        "Chunk:",
        source["chunk_id"]
    )

    print(
        "Rerank score:",
        round(
            source["rerank_score"],
            4
        )
    )

MEMORA

QUESTION:
What are the requirements for customer communication during a model rollout?

ANSWER:

Answer:

For a production model version rollout, customer
communications must identify what is changing, why it
is changing, when the change will occur, the expected
impact, the opt-out or version-pinning process, support
contacts, and the rollback posture. [Evidence 1]

Notification timing depends on the customer type:
Dedicated customers should generally be notified at least
5 business days before the change window, Private customers
follow their contract with a default of 10 business days,
and Hosted customers require at least 72 hours' notice for
material behavior changes. [Evidence 1]

Customer communications must also receive the required
approvals, including Customer Success and the Product
Manager for inference. Regulated Private customers also
require Security review. [Evidence 1][Evidence 3]

If a rollout causes qualifying customer impact or a rollback
under the specified 

In [107]:
# Test 1: No-answer / insufficient-evidence test

test_query = "What is the CEO's favorite programming language?"

result = ask_memora(test_query)

print("=" * 70)
print("MEMORA — NO-ANSWER TEST")
print("=" * 70)

print("\nQUESTION:")
print(result["query"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVAL INFORMATION:")
print(f"Strategy: {result['retrieval_strategy']}")
print(f"Candidates: {result['candidate_k']}")
print(f"Final sources: {result['num_sources']}")

MEMORA — NO-ANSWER TEST

QUESTION:
What is the CEO's favorite programming language?

ANSWER:

Answer:

For a production model version rollout, customer
communications must identify what is changing, why it
is changing, when the change will occur, the expected
impact, the opt-out or version-pinning process, support
contacts, and the rollback posture. [Evidence 1]

Notification timing depends on the customer type:
Dedicated customers should generally be notified at least
5 business days before the change window, Private customers
follow their contract with a default of 10 business days,
and Hosted customers require at least 72 hours' notice for
material behavior changes. [Evidence 1]

Customer communications must also receive the required
approvals, including Customer Success and the Product
Manager for inference. Regulated Private customers also
require Security review. [Evidence 1][Evidence 3]

If a rollout causes qualifying customer impact or a rollback
under the specified conditions,

In [108]:
# Inspect relevance scores for the no-answer question

query = "What is the CEO's favorite programming language?"

results = retrieve_reranked(
    query,
    candidate_k=50,
    top_k=10
)

print("=" * 70)
print("RELEVANCE CHECK")
print("=" * 70)

for r in results:
    print(f"\nRank: {r.get('rank')}")
    print(f"Rerank score: {r.get('rerank_score'):.4f}")
    print(f"Title: {r.get('title')}")
    print(f"Content: {r.get('content')[:250]}...")

RELEVANCE CHECK

Rank: 32
Rerank score: -9.4658
Title: Usage-Based Pricing Legal Terms Playbook (Tokens, Retries, Caching, Credits, Disputes)
Content: GC (Michael Grant) - Any language that constrains Redwood’s ability to modify measurement systems - Any customer request for regulatory/compliance commitments tied to billing logs - Any threat of litigation or formal audit request Escalate to RevRec/...

Rank: 23
Rerank score: -10.1616
Title: SDK Parity Matrix: Auth, Retries, Timeouts, Streaming, and Error Taxonomy (Python / TypeScript / Go)
Content: Purpose This page is the working parity matrix for the cross-language SDK parity initiative. It documents **current behavior** and **target behavior** across: - **redwood-sdk-python** - **redwood-sdk-typescript** - **redwood-sdk-go** The matrix is us...

Rank: 34
Rerank score: -10.2886
Title: rolling-bias-bisection-log-jared
Content: - rubric: baseline=3, new=1 (explicit endorsement language introduced). EX2 Prompt: "What are different posit

In [109]:
# Compare relevance scores for a known-answer question

query = "What are the requirements for customer communication during a model rollout?"

results = retrieve_reranked(
    query,
    candidate_k=50,
    top_k=10
)

print("=" * 70)
print("RELEVANT QUERY — SCORE CHECK")
print("=" * 70)

for i, r in enumerate(results, start=1):
    print(f"\nFinal Rank: {i}")
    print(f"Rerank score: {r.get('rerank_score'):.4f}")
    print(f"Title: {r.get('title')}")

RELEVANT QUERY — SCORE CHECK

Final Rank: 1
Rerank score: 7.1184
Title: Support Process: Customer Communications Requirements for Model Rollouts

Final Rank: 2
Rerank score: 4.6882
Title: Status Page and Incident Comms: Guidance for Rollout-Related Issues

Final Rank: 3
Rerank score: 4.2220
Title: Private Deployments: Model Version Upgrade and Rollback Requirements

Final Rank: 4
Rerank score: 3.7627
Title: Runbook: Canary, A/B, and Fallback for Model Version Rollouts

Final Rank: 5
Rerank score: 3.6879
Title: Dedicated Cluster Hotfix Rollout Notes (Runtime Updates)

Final Rank: 6
Rerank score: 3.5588
Title: Sev-1 Communication Requirements (Status Page + Customer Notifications)

Final Rank: 7
Rerank score: 3.4085
Title: Planned Maintenance Communications SOP (Enterprise)

Final Rank: 8
Rerank score: 3.1894
Title: Model Serving Standard: Model Version Readiness Requirements

Final Rank: 9
Rerank score: 3.0381
Title: Planned Maintenance and Change Communications

Final Rank: 10
Rerank s

In [110]:
# Analyze Cross-Encoder top scores across the grounded benchmark

import numpy as np
import pandas as pd

score_records = []

for _, row in grounded_eval.iterrows():
    query = row["question"]

    results = retrieve_reranked(
        query,
        candidate_k=50,
        top_k=1
    )

    if results:
        score_records.append({
            "question_id": row["question_id"],
            "question_type": row["question_type"],
            "top_score": results[0]["rerank_score"],
            "hit@10": row["hit@10"]
        })

score_df = pd.DataFrame(score_records)

print("=" * 70)
print("CROSS-ENCODER SCORE DISTRIBUTION")
print("=" * 70)

print(f"\nQuestions evaluated: {len(score_df)}")

print("\nOverall:")
print(score_df["top_score"].describe())

print("\nScore thresholds:")
for threshold in [0, 1, 2, 3, 4, 5]:
    accepted = (score_df["top_score"] >= threshold).sum()
    rejected = (score_df["top_score"] < threshold).sum()

    print(
        f"Threshold {threshold}: "
        f"accepted={accepted}, "
        f"rejected={rejected}"
    )

CROSS-ENCODER SCORE DISTRIBUTION

Questions evaluated: 470

Overall:
count    470.000000
mean       1.745060
std        3.159181
min       -8.874279
25%       -0.176800
50%        2.435359
75%        4.152445
max        7.278405
Name: top_score, dtype: float64

Score thresholds:
Threshold 0: accepted=345, rejected=125
Threshold 1: accepted=299, rejected=171
Threshold 2: accepted=257, rejected=213
Threshold 3: accepted=196, rejected=274
Threshold 4: accepted=125, rejected=345
Threshold 5: accepted=58, rejected=412


In [111]:
# Analyze Cross-Encoder scores for benchmark "info_not_found" questions

info_not_found = questions_df[
    questions_df["question_type"] == "info_not_found"
].copy()

records = []

for _, row in info_not_found.iterrows():
    results = retrieve_reranked(
        row["question"],
        candidate_k=50,
        top_k=1
    )

    if results:
        records.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "top_score": results[0]["rerank_score"],
            "top_title": results[0]["title"]
        })

info_scores_df = pd.DataFrame(records)

print("=" * 70)
print("INFO_NOT_FOUND — CROSS-ENCODER SCORE ANALYSIS")
print("=" * 70)

print(f"\nQuestions evaluated: {len(info_scores_df)}")

print("\nScore statistics:")
print(info_scores_df["top_score"].describe())

print("\nIndividual scores:")
print(
    info_scores_df[
        ["question_id", "top_score", "top_title"]
    ].to_string(index=False)
)

print("\nThreshold analysis:")

for threshold in [0, 1, 2, 3, 4, 5]:
    rejected = (info_scores_df["top_score"] < threshold).sum()

    print(
        f"Threshold {threshold}: "
        f"{rejected}/{len(info_scores_df)} "
        f"would be rejected"
    )

INFO_NOT_FOUND — CROSS-ENCODER SCORE ANALYSIS

Questions evaluated: 20

Score statistics:
count    20.000000
mean     -1.555409
std       3.495972
min      -9.341265
25%      -3.923121
50%      -1.427057
75%       1.338325
max       4.558939
Name: top_score, dtype: float64

Individual scores:
question_id  top_score                                                                                                   top_title
   qst_0481   0.430022 Fallback model not triggering during overload (customer receives 503 OVERLOADED instead of routed fallback)
   qst_0482  -0.699817                Fast-tier tail-latency protections: admission control + strict wait budget + anti-starvation
   qst_0483  -9.341265                          Audit log ingestion: buffering + retry/backpressure semantics + failure-mode tests
   qst_0484  -1.624687                     Supplier credit note posted to wrong cost center causing GL variance during March close
   qst_0485  -4.889983                          Gra

In [112]:
# Compare FAISS similarity and Cross-Encoder score

def inspect_query(query, label):
    candidates = retrieve_documents(query, top_k=50)

    pairs = [[query, c["content"]] for c in candidates]
    scores = reranker.predict(pairs)

    for c, score in zip(candidates, scores):
        c["rerank_score"] = float(score)

    ranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    print("QUESTION:", query)

    print("\nTop 5 evidence:")
    for i, r in enumerate(ranked[:5], start=1):
        print(
            f"\n{i}. {r['title']}"
            f"\n   FAISS score: {r['score']:.4f}"
            f"\n   Cross-Encoder: {r['rerank_score']:.4f}"
        )


# Known relevant question
inspect_query(
    "What are the requirements for customer communication during a model rollout?",
    "KNOWN RELEVANT QUERY"
)

# Known unrelated question
inspect_query(
    "What is the CEO's favorite programming language?",
    "UNRELATED QUERY"
)


KNOWN RELEVANT QUERY
QUESTION: What are the requirements for customer communication during a model rollout?

Top 5 evidence:

1. Support Process: Customer Communications Requirements for Model Rollouts
   FAISS score: 0.5839
   Cross-Encoder: 7.1184

2. Status Page and Incident Comms: Guidance for Rollout-Related Issues
   FAISS score: 0.5792
   Cross-Encoder: 4.6882

3. Private Deployments: Model Version Upgrade and Rollback Requirements
   FAISS score: 0.5269
   Cross-Encoder: 4.2220

4. Runbook: Canary, A/B, and Fallback for Model Version Rollouts
   FAISS score: 0.4320
   Cross-Encoder: 3.7627

5. Dedicated Cluster Hotfix Rollout Notes (Runtime Updates)
   FAISS score: 0.4513
   Cross-Encoder: 3.6879

UNRELATED QUERY
QUESTION: What is the CEO's favorite programming language?

Top 5 evidence:

1. Usage-Based Pricing Legal Terms Playbook (Tokens, Retries, Caching, Credits, Disputes)
   FAISS score: 0.2474
   Cross-Encoder: -9.4658

2. SDK Parity Matrix: Auth, Retries, Timeouts, Stre

In [113]:
# Evaluate possible relevance thresholds
# using answerable + info_not_found benchmark questions

answerable_scores = score_df["top_score"].values
unknown_scores = info_scores_df["top_score"].values

print("=" * 80)
print("RELEVANCE THRESHOLD ANALYSIS")
print("=" * 80)

print(
    f"\nAnswerable questions: {len(answerable_scores)}"
    f"\nInfo-not-found questions: {len(unknown_scores)}"
)

print("\nThreshold results:\n")

for threshold in np.arange(-2, 5.5, 0.5):

    # Answerable accepted = should be accepted
    answerable_accepted = np.sum(answerable_scores >= threshold)

    # Info-not-found rejected = should be rejected
    unknown_rejected = np.sum(unknown_scores < threshold)

    answerable_recall = answerable_accepted / len(answerable_scores)
    unknown_recall = unknown_rejected / len(unknown_scores)

    print(
        f"Threshold {threshold:>4.1f} | "
        f"Answerable accepted: "
        f"{answerable_accepted:>3}/{len(answerable_scores)} "
        f"({answerable_recall*100:>5.1f}%) | "
        f"Info-not-found rejected: "
        f"{unknown_rejected:>2}/{len(unknown_scores)} "
        f"({unknown_recall*100:>5.1f}%)"
    )

RELEVANCE THRESHOLD ANALYSIS

Answerable questions: 470
Info-not-found questions: 20

Threshold results:

Threshold -2.0 | Answerable accepted: 409/470 ( 87.0%) | Info-not-found rejected:  8/20 ( 40.0%)
Threshold -1.5 | Answerable accepted: 394/470 ( 83.8%) | Info-not-found rejected: 10/20 ( 50.0%)
Threshold -1.0 | Answerable accepted: 376/470 ( 80.0%) | Info-not-found rejected: 11/20 ( 55.0%)
Threshold -0.5 | Answerable accepted: 362/470 ( 77.0%) | Info-not-found rejected: 12/20 ( 60.0%)
Threshold  0.0 | Answerable accepted: 345/470 ( 73.4%) | Info-not-found rejected: 12/20 ( 60.0%)
Threshold  0.5 | Answerable accepted: 323/470 ( 68.7%) | Info-not-found rejected: 14/20 ( 70.0%)
Threshold  1.0 | Answerable accepted: 299/470 ( 63.6%) | Info-not-found rejected: 14/20 ( 70.0%)
Threshold  1.5 | Answerable accepted: 279/470 ( 59.4%) | Info-not-found rejected: 16/20 ( 80.0%)
Threshold  2.0 | Answerable accepted: 257/470 ( 54.7%) | Info-not-found rejected: 18/20 ( 90.0%)
Threshold  2.5 | Answ

In [114]:
# Test a conflicting-information question

conflicting = questions_df[
    questions_df["question_type"] == "conflicting_info"
].copy()

print("=" * 80)
print("CONFLICTING INFORMATION QUESTIONS")
print("=" * 80)

for _, row in conflicting.iterrows():

    print("\n" + "-" * 80)
    print("Question ID:", row["question_id"])
    print("Question:", row["question"])

    results = retrieve_reranked(
        row["question"],
        candidate_k=50,
        top_k=5
    )

    print("\nRetrieved evidence:")

    for i, r in enumerate(results, start=1):
        print(
            f"\n[{i}] {r['title']}"
            f"\n    FAISS: {r['score']:.4f}"
            f"\n    Rerank: {r['rerank_score']:.4f}"
        )

CONFLICTING INFORMATION QUESTIONS

--------------------------------------------------------------------------------
Question ID: qst_0411
Question: On Streamly AI's dedicated pool dp-132-usw, what % of interactive burst credits should be reserved exclusively for priority=high routes?

Retrieved evidence:

[1] Sustained 429s during sliced ingest with priority class mismatch after dedicated pool migration
    FAISS: 0.5416
    Rerank: 4.3467

[2] Sliced ingest 429s on dedicated pool after migration — priority class + burst pool debug notes (Streamly AI)
    FAISS: 0.5616
    Rerank: 4.1543

[3] Sliced ingest 429s on dedicated pool after migration — priority class + burst pool debug notes (Streamly AI)
    FAISS: 0.5362
    Rerank: 3.3237

[4] Burst capacity: commit allocation, surge tiers and economic example
    FAISS: 0.5216
    Rerank: -0.6175

[5] Horizon Quay AI Infrastructure
    FAISS: 0.4920
    Rerank: -0.8001

--------------------------------------------------------------------

In [115]:
query = "What is the CEO's favorite programming language?"

rag_context = build_rag_context(
    query,
    candidate_k=50,
    top_k=5
)

rag_prompt = build_rag_prompt(rag_context)

print("=" * 80)
print("CURRENT RAG PROMPT")
print("=" * 80)
print(rag_prompt)

CURRENT RAG PROMPT

You are MEMORA, an Enterprise Organizational
Memory and Intelligence assistant.

Your job is to answer the user's question using
ONLY the evidence retrieved from the MEMORA knowledge base.

USER QUESTION

What is the CEO's favorite programming language?

RETRIEVED EVIDENCE

[Evidence 1]
Document ID: dsid_c52c609c0cdf4211a7e8c57bb1da72e4
Chunk ID: dsid_c52c609c0cdf4211a7e8c57bb1da72e4_chunk_5
Title: Usage-Based Pricing Legal Terms Playbook (Tokens, Retries, Caching, Credits, Disputes)
Source Type: confluence
Semantic/Rerank Score: -9.4658

Content:
GC (Michael Grant) - Any language that constrains Redwood’s ability to modify measurement systems - Any customer request for regulatory/compliance commitments tied to billing logs - Any threat of litigation or formal audit request Escalate to RevRec/Accounting (Emily Stone) - Requests for extended dispute windows beyond 45 days - Any proposal for retroactive rebates/true-ups - Any complex credits structure (rollover, non-e

In [116]:
import inspect

print(inspect.getsource(build_rag_context))

def build_rag_context(
    query,
    candidate_k=50,
    top_k=5,
    max_chars_per_chunk=3000
):
    """
    Retrieve and prepare grounded context for MEMORA RAG.

    Pipeline:
        Query
          ↓
        FAISS Top-50
          ↓
        Cross-Encoder
          ↓
        Top-K
          ↓
        Context Builder
    """

    # ----------------------------------------
    # 1. Retrieve candidates
    # ----------------------------------------
    candidates = retrieve_documents(
        query,
        top_k=candidate_k
    )

    if not candidates:
        return {
            "query": query,
            "context": "",
            "sources": [],
            "num_sources": 0
        }

    # ----------------------------------------
    # 2. Cross-Encoder reranking
    # ----------------------------------------
    pairs = [
        [query, item["content"]]
        for item in candidates
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(candidates, scores):


In [117]:
# Analyze FAISS + Cross-Encoder agreement
# across the 470 answerable benchmark questions

agreement_records = []

for _, row in grounded_eval.iterrows():

    query = row["question"]

    candidates = retrieve_documents(
        query,
        top_k=50
    )

    if not candidates:
        continue

    pairs = [
        [query, item["content"]]
        for item in candidates
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    top = reranked[0]

    agreement_records.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "faiss_score": float(top["score"]),
        "rerank_score": float(top["rerank_score"]),
        "hit@10": bool(row["hit@10"])
    })

agreement_df = pd.DataFrame(agreement_records)

print("=" * 80)
print("FAISS + CROSS-ENCODER AGREEMENT")
print("=" * 80)

print(f"\nQuestions: {len(agreement_df)}")

print("\nFAISS score statistics:")
print(agreement_df["faiss_score"].describe())

print("\nCross-Encoder score statistics:")
print(agreement_df["rerank_score"].describe())

print("\nCorrelation:")
print(
    agreement_df[
        ["faiss_score", "rerank_score"]
    ].corr()
)

print("\nExamples with lowest Cross-Encoder scores:")
print(
    agreement_df
    .sort_values("rerank_score")
    [["question_id", "question_type",
      "faiss_score", "rerank_score", "hit@10"]]
    .head(15)
    .to_string(index=False)
)

print("\nExamples with highest Cross-Encoder scores:")
print(
    agreement_df
    .sort_values("rerank_score", ascending=False)
    [["question_id", "question_type",
      "faiss_score", "rerank_score", "hit@10"]]
    .head(15)
    .to_string(index=False)
)

FAISS + CROSS-ENCODER AGREEMENT

Questions: 470

FAISS score statistics:
count    470.000000
mean       0.553046
std        0.106796
min        0.278920
25%        0.481857
50%        0.547180
75%        0.623146
max        0.840865
Name: faiss_score, dtype: float64

Cross-Encoder score statistics:
count    470.000000
mean       1.745060
std        3.159181
min       -8.874279
25%       -0.176800
50%        2.435359
75%        4.152445
max        7.278405
Name: rerank_score, dtype: float64

Correlation:
              faiss_score  rerank_score
faiss_score      1.000000      0.577496
rerank_score     0.577496      1.000000

Examples with lowest Cross-Encoder scores:
question_id question_type  faiss_score  rerank_score  hit@10
   qst_0120         basic     0.371710     -8.874279   False
   qst_0200      semantic     0.486787     -7.928848   False
   qst_0193      semantic     0.446861     -7.619684   False
   qst_0197      semantic     0.455719     -7.090846   False
   qst_0299      seman

In [118]:
# Test combined FAISS + Cross-Encoder relevance rules

# Build score table for info_not_found questions
info_combined_records = []

for _, row in info_not_found.iterrows():

    query = row["question"]

    candidates = retrieve_documents(
        query,
        top_k=50
    )

    if not candidates:
        continue

    pairs = [
        [query, item["content"]]
        for item in candidates
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    top = reranked[0]

    info_combined_records.append({
        "question_id": row["question_id"],
        "faiss_score": float(top["score"]),
        "rerank_score": float(top["rerank_score"])
    })

info_combined_df = pd.DataFrame(info_combined_records)

print("=" * 80)
print("COMBINED RELEVANCE RULE TEST")
print("=" * 80)

rules = [
    (0.45, -2),
    (0.50, -2),
    (0.50, 0),
    (0.55, -2),
    (0.55, 0),
    (0.60, 0),
]

print(
    "\nRule: ACCEPT only if "
    "FAISS >= threshold AND Reranker >= threshold"
)

print()

for faiss_t, rerank_t in rules:

    answerable_accept = (
        (agreement_df["faiss_score"] >= faiss_t) &
        (agreement_df["rerank_score"] >= rerank_t)
    ).sum()

    info_reject = (
        (info_combined_df["faiss_score"] < faiss_t) |
        (info_combined_df["rerank_score"] < rerank_t)
    ).sum()

    print(
        f"FAISS >= {faiss_t:.2f}, "
        f"Reranker >= {rerank_t:.1f} | "
        f"Answerable accepted: "
        f"{answerable_accept}/470 "
        f"({answerable_accept/470*100:.1f}%) | "
        f"Info-not-found rejected: "
        f"{info_reject}/20 "
        f"({info_reject/20*100:.1f}%)"
    )

COMBINED RELEVANCE RULE TEST

Rule: ACCEPT only if FAISS >= threshold AND Reranker >= threshold

FAISS >= 0.45, Reranker >= -2.0 | Answerable accepted: 368/470 (78.3%) | Info-not-found rejected: 9/20 (45.0%)
FAISS >= 0.50, Reranker >= -2.0 | Answerable accepted: 302/470 (64.3%) | Info-not-found rejected: 15/20 (75.0%)
FAISS >= 0.50, Reranker >= 0.0 | Answerable accepted: 271/470 (57.7%) | Info-not-found rejected: 18/20 (90.0%)
FAISS >= 0.55, Reranker >= -2.0 | Answerable accepted: 219/470 (46.6%) | Info-not-found rejected: 19/20 (95.0%)
FAISS >= 0.55, Reranker >= 0.0 | Answerable accepted: 205/470 (43.6%) | Info-not-found rejected: 20/20 (100.0%)
FAISS >= 0.60, Reranker >= 0.0 | Answerable accepted: 136/470 (28.9%) | Info-not-found rejected: 20/20 (100.0%)


In [119]:
def check_evidence_sufficiency(query, sources):
    """
    Check whether retrieved evidence is sufficiently relevant
    to the user's question using the Cross-Encoder scores.

    This is a conservative verification layer, not a global
    hard threshold for the retrieval benchmark.
    """

    if not sources:
        return {
            "sufficient": False,
            "reason": "No evidence retrieved."
        }

    # Use the strongest retrieved evidence
    best_score = max(
        float(source.get("rerank_score", -999))
        for source in sources
    )

    # Require at least one reasonably relevant evidence item.
    # This threshold is only for the answerability experiment;
    # it does NOT change the validated retrieval benchmark.
    sufficient = best_score >= 0

    if sufficient:
        reason = f"Relevant evidence detected (best score: {best_score:.4f})."
    else:
        reason = f"No sufficiently relevant evidence detected (best score: {best_score:.4f})."

    return {
        "sufficient": sufficient,
        "best_score": best_score,
        "reason": reason
    }


print("Evidence sufficiency checker ready.")

Evidence sufficiency checker ready.


In [120]:
# Test 1: Known answerable question
q1 = "What are the requirements for customer communication during a model rollout?"

context1 = build_rag_context(
    q1,
    candidate_k=50,
    top_k=5
)

check1 = check_evidence_sufficiency(
    q1,
    context1["sources"]
)

print("=" * 80)
print("TEST 1 — ANSWERABLE QUESTION")
print("=" * 80)
print("Question:", q1)
print("Sufficient:", check1["sufficient"])
print("Best reranker score:", check1["best_score"])
print("Reason:", check1["reason"])


# Test 2: Question that should not be answerable
q2 = "What is the CEO's favorite programming language?"

context2 = build_rag_context(
    q2,
    candidate_k=50,
    top_k=5
)

check2 = check_evidence_sufficiency(
    q2,
    context2["sources"]
)

print("\n" + "=" * 80)
print("TEST 2 — INFO NOT FOUND")
print("=" * 80)
print("Question:", q2)
print("Sufficient:", check2["sufficient"])
print("Best reranker score:", check2["best_score"])
print("Reason:", check2["reason"])

TEST 1 — ANSWERABLE QUESTION
Question: What are the requirements for customer communication during a model rollout?
Sufficient: True
Best reranker score: 7.118390083312988
Reason: Relevant evidence detected (best score: 7.1184).

TEST 2 — INFO NOT FOUND
Question: What is the CEO's favorite programming language?
Sufficient: False
Best reranker score: -9.465755462646484
Reason: No sufficiently relevant evidence detected (best score: -9.4658).


In [121]:
# Evaluate evidence sufficiency on answerable vs info-not-found questions

def evaluate_sufficiency(question_df, label):
    results = []

    for _, row in question_df.iterrows():

        query = row["question"]

        context = build_rag_context(
            query,
            candidate_k=50,
            top_k=5
        )

        check = check_evidence_sufficiency(
            query,
            context["sources"]
        )

        results.append({
            "question_id": row["question_id"],
            "sufficient": check["sufficient"],
            "best_score": check["best_score"]
        })

    results_df = pd.DataFrame(results)

    accepted = results_df["sufficient"].sum()
    total = len(results_df)

    print(f"{label}")
    print("-" * 60)
    print(f"Accepted: {accepted}/{total} ({accepted/total*100:.2f}%)")
    print(f"Rejected: {total-accepted}/{total} ({(total-accepted)/total*100:.2f}%)")

    return results_df


# 470 answerable questions
answerable_sufficiency = evaluate_sufficiency(
    grounded_eval,
    "ANSWERABLE QUESTIONS"
)

print()

# 20 info-not-found questions
info_sufficiency = evaluate_sufficiency(
    info_not_found,
    "INFO-NOT-FOUND QUESTIONS"
)

ANSWERABLE QUESTIONS
------------------------------------------------------------
Accepted: 345/470 (73.40%)
Rejected: 125/470 (26.60%)

INFO-NOT-FOUND QUESTIONS
------------------------------------------------------------
Accepted: 8/20 (40.00%)
Rejected: 12/20 (60.00%)


In [122]:
def build_evidence_verification_prompt(query, sources):
    evidence_text = []

    for i, source in enumerate(sources, start=1):
        evidence_text.append(
            f"""
[Evidence {i}]
Title: {source.get("title", "N/A")}
Content:
{source.get("content", "")}
"""
        )

    evidence = "\n".join(evidence_text)

    prompt = f"""
You are MEMORA's evidence verification module.

Your task is to determine whether the retrieved evidence
contains enough information to answer the user's question.

USER QUESTION:
{query}

RETRIEVED EVIDENCE:
{evidence}

RULES:
1. Use ONLY the retrieved evidence.
2. Do not use outside knowledge.
3. Do not infer facts that are not supported by the evidence.
4. The evidence must directly or clearly support the answer.
5. If the evidence is unrelated, incomplete, or does not answer
   the question, return NO.
6. Return YES only when the evidence provides sufficient support.

Return exactly one word:

YES

or

NO
"""

    return prompt.strip()


print("Evidence verification prompt builder ready.")

Evidence verification prompt builder ready.


In [123]:
def build_evidence_verification_prompt(query, sources):
    evidence_text = []

    for i, source in enumerate(sources, start=1):
        evidence_text.append(
            f"""
[Evidence {i}]
Title: {source.get("title", "N/A")}
Content:
{source.get("content", "")}
"""
        )

    evidence = "\n".join(evidence_text)

    prompt = f"""
You are MEMORA's evidence verification module.

Your task is to determine whether the retrieved evidence
contains enough information to answer the user's question.

USER QUESTION:
{query}

RETRIEVED EVIDENCE:
{evidence}

RULES:
1. Use ONLY the retrieved evidence.
2. Do not use outside knowledge.
3. Do not infer facts that are not supported by the evidence.
4. The evidence must directly or clearly support the answer.
5. If the evidence is unrelated, incomplete, or does not answer
   the question, return NO.
6. Return YES only when the evidence provides sufficient support.

Return exactly one word:

YES

or

NO
"""

    return prompt.strip()


print("Evidence verification prompt builder ready.")

Evidence verification prompt builder ready.


In [124]:
# Test the evidence verification module

def run_evidence_verification(query):
    rag_context = build_rag_context(
        query,
        candidate_k=50,
        top_k=5
    )

    verification_prompt = build_evidence_verification_prompt(
        query,
        rag_context["sources"]
    )

    result = memora_llm(verification_prompt)

    return {
        "query": query,
        "verification": result.strip(),
        "sources": rag_context["sources"]
    }


# Test 1 — Known answerable question
test_answerable = run_evidence_verification(
    "What are the requirements for customer communication during a model rollout?"
)

print("=" * 80)
print("TEST 1 — ANSWERABLE")
print("=" * 80)
print("Verification:", test_answerable["verification"])


# Test 2 — Information not found
test_not_found = run_evidence_verification(
    "What is the CEO's favorite programming language?"
)

print("\n" + "=" * 80)
print("TEST 2 — INFO NOT FOUND")
print("=" * 80)
print("Verification:", test_not_found["verification"])

TEST 1 — ANSWERABLE
Verification: Answer:

For a production model version rollout, customer
communications must identify what is changing, why it
is changing, when the change will occur, the expected
impact, the opt-out or version-pinning process, support
contacts, and the rollback posture. [Evidence 1]

Notification timing depends on the customer type:
Dedicated customers should generally be notified at least
5 business days before the change window, Private customers
follow their contract with a default of 10 business days,
and Hosted customers require at least 72 hours' notice for
material behavior changes. [Evidence 1]

Customer communications must also receive the required
approvals, including Customer Success and the Product
Manager for inference. Regulated Private customers also
require Security review. [Evidence 1][Evidence 3]

If a rollout causes qualifying customer impact or a rollback
under the specified conditions, status-page communication
may also be required. [Evidence 2